<a href="https://colab.research.google.com/github/BuruhArloji/home_credit_prediction/blob/main/Home_Credit_Loan_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Problem**

Lembaga keuangan pemberi pinjaman menghadapi risiko finansial yang signifikan ketika memberikan kredit kepada individu yang memiliki potensi tinggi untuk gagal bayar (default). Proses peninjauan manual untuk menentukan kelayakan kredit sering kali memakan waktu, subjektif, dan tidak efisien dalam memproses ribuan aplikasi secara cepat.

Ketidakmampuan dalam membedakan antara peminjam yang dapat dipercaya dan peminjam berisiko tinggi dapat menyebabkan dua kerugian:

* Kerugian Finansial: Memberikan pinjaman kepada orang yang akhirnya gagal bayar.

* Kehilangan Kesempatan: Menolak peminjam yang sebenarnya mampu membayar, sehingga perusahaan kehilangan potensi keuntungan.

# **Objective**

Mengembangkan model Machine Learning klasifikasi biner yang dapat memprediksi probabilitas seorang pemohon pinjaman akan mengalami kesulitan pembayaran di masa depan.

Secara spesifik, tujuan proyek ini adalah:
* Membangun Model Prediktif: Menggunakan algoritma klasifikasi untuk membedakan antara kelas Target 0 dan Target 1.

* Identifikasi Indikator Kunci: Menentukan fitur-fitur apa saja (misal: AMT_INCOME_TOTAL, DAYS_BIRTH, EXT_SOURCE) yang paling berpengaruh terhadap kemungkinan gagal bayar.

* Optimasi Keputusan Bisnis: Menyediakan alat bantu bagi tim analis kredit untuk menyaring aplikasi secara otomatis dengan tingkat akurasi dan recall yang tinggi, guna meminimalkan risiko kerugian kredit.

# **1.&nbsp;Setup and Initialization**

## 1.1. Loading Datasets

In [ ]:
from google.colab import drive

# Menghubungkan Google Drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



### 1.1.1 Application

In [ ]:
path_train = '/content/drive/MyDrive/Home Credit Risk/application_train.csv'
df_application = pd.read_csv(path_train)
df_application.head()

### 1.1.2 Bureau

In [ ]:
path_bureau = '/content/drive/MyDrive/Home Credit Risk/bureau.csv'
df_bureau = pd.read_csv(path_bureau)
df_bureau.head()

### 1.1.3 Previous Application

In [ ]:
path_previousapp = '/content/drive/MyDrive/Home Credit Risk/previous_application.csv'
df_previousapp = pd.read_csv(path_previousapp)
df_previousapp.head()

### 1.1.4 Bureau Balance

In [ ]:
path_bureaubalance = '/content/drive/MyDrive/Home Credit Risk/bureau_balance.csv'
df_bureaubalance = pd.read_csv(path_bureaubalance)
df_bureaubalance.head()

### 1.1.5 POS Cash Balance

In [ ]:
path_poscashbalance = '/content/drive/MyDrive/Home Credit Risk/POS_CASH_balance.csv'
df_poscashbalance = pd.read_csv(path_poscashbalance)
df_poscashbalance.head()

### 1.1.6 Installments Payment

In [ ]:
path_installments = '/content/drive/MyDrive/Home Credit Risk/installments_payments.csv'
df_installments = pd.read_csv(path_installments)
df_installments.head()

### 1.1.7 Credit Card Balance

In [ ]:
path_credit_card_balance = '/content/drive/MyDrive/Home Credit Risk/credit_card_balance.csv'
df_credit_card_balance = pd.read_csv(path_credit_card_balance)
df_credit_card_balance.head()

## 1.2. Data Description

### 1.2.1 Application



| **Variable** | **Description** |
| :--- | :--- |
| **SK_ID_CURR** | Identifier unik yang diberikan kepada setiap aplikasi pinjaman, digunakan untuk membedakan masing-masing pemohon. |
| **TARGET** | Variabel target biner yang menunjukkan apakah pemohon mengalami kesulitan pembayaran (1 = ya, 0 = tidak). |
| **NAME_CONTRACT_TYPE** | Jenis kontrak pinjaman yang diajukan, misalnya pinjaman tunai (*Cash loans*) atau pinjaman bergulir (*Revolving loans*). |
| **CODE_GENDER** | Jenis kelamin pemohon, direpresentasikan sebagai kode kategori (misalnya M untuk pria, F untuk wanita). |
| **FLAG_OWN_CAR** | Indikator biner yang menunjukkan apakah pemohon memiliki kendaraan (Y = ya, N = tidak). |
| **FLAG_OWN_REALTY** | Indikator biner yang menunjukkan apakah pemohon memiliki properti atau real estat (Y = ya, N = tidak). |
| **CNT_CHILDREN** | Jumlah anak yang dimiliki oleh pemohon. |
| **AMT_INCOME_TOTAL** | Total pendapatan tahunan pemohon dalam satuan mata uang. |
| **AMT_CREDIT** | Jumlah total kredit yang diajukan oleh pemohon. |
| **AMT_ANNUITY** | Nilai anuitas pinjaman, yaitu jumlah cicilan yang harus dibayar secara berkala. |
| **AMT_GOODS_PRICE** | Harga barang atau properti yang menjadi tujuan pembiayaan dari kredit yang diajukan. |
| **NAME_TYPE_SUITE** | Keterangan mengenai siapa yang menemani pemohon saat mengajukan kredit (misalnya tidak ada pendamping, pasangan, anak, dll.). |
| **NAME_INCOME_TYPE** | Jenis sumber pendapatan pemohon, misalnya karyawan (*Working*), pensiunan (*Pensioner*), wiraswasta (*Self-employed*), dll. |
| **NAME_EDUCATION_TYPE** | Tingkat pendidikan tertinggi yang ditempuh oleh pemohon. |
| **NAME_FAMILY_STATUS** | Status pernikahan atau keluarga pemohon, misalnya menikah, lajang, cerai, dll. |
| **NAME_HOUSING_TYPE** | Jenis tempat tinggal pemohon, misalnya rumah/apartemen milik sendiri, sewa, tinggal bersama orang tua, dll. |
| **REGION_POPULATION_RELATIVE** | Kepadatan populasi relatif di wilayah tempat tinggal pemohon, dinormalisasi terhadap seluruh wilayah. |
| **DAYS_BIRTH** | Usia pemohon dalam satuan hari (nilai negatif, dihitung mundur dari tanggal pengajuan). |
| **DAYS_EMPLOYED** | Jumlah hari pemohon telah bekerja di pekerjaan saat ini (nilai negatif menunjukkan masa lalu; nilai positif besar seperti 365243 menandakan pengangguran atau tidak bekerja). |
| **DAYS_REGISTRATION** | Jumlah hari sejak pemohon terakhir melakukan registrasi dokumen identitas (nilai negatif). |
| **DAYS_ID_PUBLISH** | Jumlah hari sejak pemohon terakhir memperbarui dokumen identitasnya (nilai negatif). |
| **OWN_CAR_AGE** | Usia kendaraan yang dimiliki pemohon, dalam satuan tahun. Hanya terisi jika pemohon memiliki kendaraan. |
| **FLAG_MOBIL** | Indikator biner apakah pemohon menyediakan nomor telepon seluler (1 = ya, 0 = tidak). |
| **FLAG_EMP_PHONE** | Indikator biner apakah pemohon menyediakan nomor telepon kantor/tempat kerja (1 = ya, 0 = tidak). |
| **FLAG_WORK_PHONE** | Indikator biner apakah pemohon menyediakan nomor telepon kerja alternatif (1 = ya, 0 = tidak). |
| **FLAG_CONT_MOBILE** | Indikator biner apakah nomor telepon seluler pemohon dapat dihubungi (1 = ya, 0 = tidak). |
| **FLAG_PHONE** | Indikator biner apakah pemohon menyediakan nomor telepon rumah (1 = ya, 0 = tidak). |
| **FLAG_EMAIL** | Indikator biner apakah pemohon menyediakan alamat email (1 = ya, 0 = tidak). |
| **OCCUPATION_TYPE** | Jenis pekerjaan atau profesi yang dijalankan oleh pemohon. |
| **CNT_FAM_MEMBERS** | Jumlah total anggota keluarga yang dimiliki oleh pemohon. |
| **REGION_RATING_CLIENT** | Penilaian atau rating wilayah tempat tinggal pemohon yang diberikan oleh perusahaan (skala 1–3). |
| **REGION_RATING_CLIENT_W_CITY** | Penilaian wilayah tempat tinggal pemohon dengan mempertimbangkan kota, yang diberikan oleh perusahaan (skala 1–3). |
| **WEEKDAY_APPR_PROCESS_START** | Hari dalam seminggu ketika proses pengajuan kredit dimulai (misalnya Monday, Tuesday, dll.). |
| **HOUR_APPR_PROCESS_START** | Jam saat proses pengajuan kredit dimulai, dalam format 24 jam. |
| **REG_REGION_NOT_LIVE_REGION** | Indikator biner apakah wilayah registrasi pemohon berbeda dengan wilayah tempat tinggalnya (1 = berbeda, 0 = sama). |
| **REG_REGION_NOT_WORK_REGION** | Indikator biner apakah wilayah registrasi pemohon berbeda dengan wilayah tempat bekerjanya (1 = berbeda, 0 = sama). |
| **LIVE_REGION_NOT_WORK_REGION** | Indikator biner apakah wilayah tempat tinggal pemohon berbeda dengan wilayah tempat bekerjanya (1 = berbeda, 0 = sama). |
| **REG_CITY_NOT_LIVE_CITY** | Indikator biner apakah kota registrasi pemohon berbeda dengan kota tempat tinggalnya (1 = berbeda, 0 = sama). |
| **REG_CITY_NOT_WORK_CITY** | Indikator biner apakah kota registrasi pemohon berbeda dengan kota tempat bekerjanya (1 = berbeda, 0 = sama). |
| **LIVE_CITY_NOT_WORK_CITY** | Indikator biner apakah kota tempat tinggal pemohon berbeda dengan kota tempat bekerjanya (1 = berbeda, 0 = sama). |
| **ORGANIZATION_TYPE** | Jenis organisasi atau industri tempat pemohon bekerja. |
| **EXT_SOURCE_1** | Skor atau nilai terstandarisasi dari sumber data eksternal pertama, digunakan sebagai indikator kelayakan kredit. |
| **EXT_SOURCE_2** | Skor atau nilai terstandarisasi dari sumber data eksternal kedua, digunakan sebagai indikator kelayakan kredit. |
| **EXT_SOURCE_3** | Skor atau nilai terstandarisasi dari sumber data eksternal ketiga, digunakan sebagai indikator kelayakan kredit. |
| **APARTMENTS_AVG** | Nilai rata-rata (*average*) dari karakteristik apartemen di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **BASEMENTAREA_AVG** | Nilai rata-rata luas area basement di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **YEARS_BEGINEXPLUATATION_AVG** | Rata-rata tahun mulai beroperasinya bangunan di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **YEARS_BUILD_AVG** | Rata-rata tahun pembangunan gedung di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **COMMONAREA_AVG** | Nilai rata-rata luas area umum bersama di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **ELEVATORS_AVG** | Nilai rata-rata jumlah lift di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **ENTRANCES_AVG** | Nilai rata-rata jumlah pintu masuk di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **FLOORSMAX_AVG** | Nilai rata-rata jumlah lantai tertinggi di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **FLOORSMIN_AVG** | Nilai rata-rata jumlah lantai terendah di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **LANDAREA_AVG** | Nilai rata-rata luas lahan di sekitar tempat tinggal pemohon, dinormalisasi. |
| **LIVINGAPARTMENTS_AVG** | Nilai rata-rata jumlah unit hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **LIVINGAREA_AVG** | Nilai rata-rata luas area hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **NONLIVINGAPARTMENTS_AVG** | Nilai rata-rata jumlah unit non-hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **NONLIVINGAREA_AVG** | Nilai rata-rata luas area non-hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **APARTMENTS_MODE** | Nilai modus (*mode*) dari karakteristik apartemen di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **BASEMENTAREA_MODE** | Nilai modus luas area basement di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **YEARS_BEGINEXPLUATATION_MODE** | Nilai modus tahun mulai beroperasinya bangunan di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **YEARS_BUILD_MODE** | Nilai modus tahun pembangunan gedung di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **COMMONAREA_MODE** | Nilai modus luas area umum bersama di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **ELEVATORS_MODE** | Nilai modus jumlah lift di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **ENTRANCES_MODE** | Nilai modus jumlah pintu masuk di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **FLOORSMAX_MODE** | Nilai modus jumlah lantai tertinggi di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **FLOORSMIN_MODE** | Nilai modus jumlah lantai terendah di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **LANDAREA_MODE** | Nilai modus luas lahan di sekitar tempat tinggal pemohon, dinormalisasi. |
| **LIVINGAPARTMENTS_MODE** | Nilai modus jumlah unit hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **LIVINGAREA_MODE** | Nilai modus luas area hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **NONLIVINGAPARTMENTS_MODE** | Nilai modus jumlah unit non-hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **NONLIVINGAREA_MODE** | Nilai modus luas area non-hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **APARTMENTS_MEDI** | Nilai median (*median*) dari karakteristik apartemen di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **BASEMENTAREA_MEDI** | Nilai median luas area basement di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **YEARS_BEGINEXPLUATATION_MEDI** | Nilai median tahun mulai beroperasinya bangunan di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **YEARS_BUILD_MEDI** | Nilai median tahun pembangunan gedung di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **COMMONAREA_MEDI** | Nilai median luas area umum bersama di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **ELEVATORS_MEDI** | Nilai median jumlah lift di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **ENTRANCES_MEDI** | Nilai median jumlah pintu masuk di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **FLOORSMAX_MEDI** | Nilai median jumlah lantai tertinggi di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **FLOORSMIN_MEDI** | Nilai median jumlah lantai terendah di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **LANDAREA_MEDI** | Nilai median luas lahan di sekitar tempat tinggal pemohon, dinormalisasi. |
| **LIVINGAPARTMENTS_MEDI** | Nilai median jumlah unit hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **LIVINGAREA_MEDI** | Nilai median luas area hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **NONLIVINGAPARTMENTS_MEDI** | Nilai median jumlah unit non-hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **NONLIVINGAREA_MEDI** | Nilai median luas area non-hunian di bangunan sekitar tempat tinggal pemohon, dinormalisasi. |
| **FONDKAPREMONT_MODE** | Modus jenis sumber dana perbaikan bangunan di lingkungan tempat tinggal pemohon (kategorikal). |
| **HOUSETYPE_MODE** | Modus jenis bangunan tempat tinggal pemohon (kategorikal), misalnya blok, rumah terpisah, dll. |
| **TOTALAREA_MODE** | Nilai modus total luas keseluruhan bangunan di lingkungan tempat tinggal pemohon, dinormalisasi. |
| **WALLSMATERIAL_MODE** | Modus jenis material dinding bangunan di lingkungan tempat tinggal pemohon (kategorikal). |
| **EMERGENCYSTATE_MODE** | Modus status kondisi darurat bangunan di lingkungan tempat tinggal pemohon (kategorikal, misalnya Yes/No). |
| **OBS_30_CNT_SOCIAL_CIRCLE** | Jumlah orang dalam lingkaran sosial pemohon yang pernah mengalami keterlambatan pembayaran lebih dari 30 hari. |
| **DEF_30_CNT_SOCIAL_CIRCLE** | Jumlah orang dalam lingkaran sosial pemohon yang mengalami gagal bayar (*default*) lebih dari 30 hari. |
| **OBS_60_CNT_SOCIAL_CIRCLE** | Jumlah orang dalam lingkaran sosial pemohon yang pernah mengalami keterlambatan pembayaran lebih dari 60 hari. |
| **DEF_60_CNT_SOCIAL_CIRCLE** | Jumlah orang dalam lingkaran sosial pemohon yang mengalami gagal bayar (*default*) lebih dari 60 hari. |
| **DAYS_LAST_PHONE_CHANGE** | Jumlah hari sejak pemohon terakhir mengganti nomor teleponnya (nilai negatif). |
| **FLAG_DOCUMENT_2** s/d **FLAG_DOCUMENT_21** | Serangkaian indikator biner yang menunjukkan apakah pemohon menyerahkan dokumen tertentu bernomor 2 hingga 21 saat pengajuan (1 = diserahkan, 0 = tidak diserahkan). |
| **AMT_REQ_CREDIT_BUREAU_HOUR** | Jumlah permintaan informasi kredit (*credit bureau enquiry*) terhadap pemohon dalam satu jam terakhir sebelum pengajuan. |
| **AMT_REQ_CREDIT_BUREAU_DAY** | Jumlah permintaan informasi kredit terhadap pemohon dalam satu hari terakhir sebelum pengajuan. |
| **AMT_REQ_CREDIT_BUREAU_WEEK** | Jumlah permintaan informasi kredit terhadap pemohon dalam satu minggu terakhir sebelum pengajuan. |
| **AMT_REQ_CREDIT_BUREAU_MON** | Jumlah permintaan informasi kredit terhadap pemohon dalam satu bulan terakhir sebelum pengajuan. |
| **AMT_REQ_CREDIT_BUREAU_QRT** | Jumlah permintaan informasi kredit terhadap pemohon dalam satu kuartal (tiga bulan) terakhir sebelum pengajuan. |
| **AMT_REQ_CREDIT_BUREAU_YEAR** | Jumlah permintaan informasi kredit terhadap pemohon dalam satu tahun terakhir sebelum pengajuan. |
c

### 1.2.2 Bureau


| **Variable**               | **Description**                                                                                                                               |
| :------------------------- | :-------------------------------------------------------------------------------------------------------------------------------------------- |
| **SK_ID_CURR**             | Identifier unik untuk setiap nasabah (client) yang mengajukan kredit. Digunakan sebagai key untuk menghubungkan ke tabel utama (application). |
| **SK_ID_BUREAU**           | Identifier unik untuk setiap catatan kredit nasabah di lembaga kredit eksternal (bureau). Satu nasabah bisa memiliki banyak ID ini.           |
| **CREDIT_ACTIVE**          | Status kredit di bureau, misalnya *Active* (masih berjalan), *Closed* (sudah lunas), dll.                                                     |
| **CREDIT_CURRENCY**        | Jenis mata uang dari kredit yang tercatat (misalnya currency 1, currency 2, dll).                                                             |
| **DAYS_CREDIT**            | Jumlah hari sejak kredit tersebut diberikan (nilai negatif berarti terjadi di masa lalu sebelum aplikasi saat ini).                           |
| **CREDIT_DAY_OVERDUE**     | Jumlah hari keterlambatan pembayaran kredit (overdue). Nilai 0 berarti tidak ada keterlambatan.                                               |
| **DAYS_CREDIT_ENDDATE**    | Jumlah hari hingga tanggal akhir kontrak kredit yang direncanakan (nilai negatif berarti sudah lewat dari tanggal akhir).                     |
| **DAYS_ENDDATE_FACT**      | Jumlah hari hingga tanggal sebenarnya kredit berakhir (jika sudah selesai).                                                                   |
| **AMT_CREDIT_MAX_OVERDUE** | Jumlah maksimum keterlambatan pembayaran yang pernah terjadi pada kredit tersebut.                                                            |
| **CNT_CREDIT_PROLONG**     | Jumlah perpanjangan kredit yang pernah dilakukan oleh nasabah untuk kredit ini.                                                               |
| **AMT_CREDIT_SUM**         | Total jumlah kredit yang diberikan pada kontrak tersebut.                                                                                     |
| **AMT_CREDIT_SUM_DEBT**    | Jumlah sisa utang yang masih harus dibayar oleh nasabah untuk kredit ini.                                                                     |
| **AMT_CREDIT_SUM_LIMIT**   | Batas maksimum kredit yang tersedia (biasanya untuk kredit jenis revolving seperti kartu kredit).                                             |
| **AMT_CREDIT_SUM_OVERDUE** | Jumlah utang yang sudah jatuh tempo dan belum dibayar (overdue amount).                                                                       |
| **CREDIT_TYPE**            | Jenis kredit, misalnya *Consumer credit*, *Credit card*, dll.                                                                                 |
| **DAYS_CREDIT_UPDATE**     | Jumlah hari sejak terakhir kali data kredit ini diperbarui oleh bureau (nilai negatif berarti di masa lalu).                                  |
| **AMT_ANNUITY**            | Nilai cicilan (anuitas) yang harus dibayar secara berkala untuk kredit ini.                                                                   |


### 1.2.3 Previous Application

| **Variable**                    | **Description**                                                                                                                                               |
| :------------------------------ | :------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **SK_ID_PREV**                  | Identifier unik untuk setiap aplikasi kredit sebelumnya yang pernah diajukan oleh nasabah.                                                                    |
| **SK_ID_CURR**                  | Identifier unik nasabah, digunakan untuk menghubungkan data ini dengan tabel utama aplikasi.                                                                  |
| **NAME_CONTRACT_TYPE**          | Jenis kontrak kredit pada aplikasi sebelumnya, misalnya *Cash loans*, *Consumer loans*, atau *Revolving loans*.                                               |
| **AMT_ANNUITY**                 | Jumlah cicilan atau anuitas yang harus dibayar secara berkala untuk aplikasi kredit sebelumnya.                                                               |
| **AMT_APPLICATION**             | Jumlah kredit yang diajukan oleh nasabah pada aplikasi sebelumnya.                                                                                            |
| **AMT_CREDIT**                  | Jumlah kredit yang disetujui oleh pemberi pinjaman pada aplikasi sebelumnya.                                                                                  |
| **AMT_DOWN_PAYMENT**            | Jumlah uang muka yang dibayarkan oleh nasabah saat pengajuan kredit.                                                                                          |
| **AMT_GOODS_PRICE**             | Harga barang atau aset yang dibiayai oleh kredit tersebut.                                                                                                    |
| **WEEKDAY_APPR_PROCESS_START**  | Hari dalam seminggu saat proses aplikasi kredit dimulai.                                                                                                      |
| **HOUR_APPR_PROCESS_START**     | Jam saat proses aplikasi kredit dimulai, dalam format 24 jam.                                                                                                 |
| **FLAG_LAST_APPL_PER_CONTRACT** | Penanda apakah aplikasi tersebut merupakan aplikasi terakhir untuk kontrak kredit terkait.                                                                    |
| **NFLAG_LAST_APPL_IN_DAY**      | Indikator apakah aplikasi tersebut merupakan aplikasi terakhir yang diajukan nasabah pada hari yang sama.                                                     |
| **RATE_DOWN_PAYMENT**           | Rasio uang muka terhadap nilai aplikasi atau harga barang yang dibiayai.                                                                                      |
| **RATE_INTEREST_PRIMARY**       | Tingkat bunga utama atau bunga standar yang berlaku pada kredit sebelumnya.                                                                                   |
| **RATE_INTEREST_PRIVILEGED**    | Tingkat bunga khusus atau preferensial yang diberikan kepada nasabah tertentu.                                                                                |
| **NAME_CASH_LOAN_PURPOSE**      | Tujuan pengajuan pinjaman tunai, misalnya pendidikan, renovasi, pembelian barang, atau keperluan lainnya.                                                     |
| **NAME_CONTRACT_STATUS**        | Status akhir aplikasi kredit sebelumnya, misalnya *Approved*, *Rejected*, *Canceled*, atau *Unused offer*.                                                    |
| **DAYS_DECISION**               | Jumlah hari relatif terhadap tanggal aplikasi saat ini ketika keputusan kredit sebelumnya dibuat. Nilai negatif berarti keputusan terjadi di masa lalu.       |
| **NAME_PAYMENT_TYPE**           | Jenis metode pembayaran yang digunakan untuk aplikasi kredit sebelumnya.                                                                                      |
| **CODE_REJECT_REASON**          | Kode alasan penolakan jika aplikasi kredit sebelumnya ditolak.                                                                                                |
| **NAME_TYPE_SUITE**             | Informasi siapa yang menemani nasabah saat mengajukan aplikasi kredit, misalnya keluarga, pasangan, atau tidak ditemani.                                      |
| **NAME_CLIENT_TYPE**            | Jenis nasabah berdasarkan riwayat hubungannya dengan lembaga pemberi kredit, misalnya nasabah baru atau lama.                                                 |
| **NAME_GOODS_CATEGORY**         | Kategori barang yang dibiayai oleh kredit, misalnya elektronik, kendaraan, furnitur, atau kategori lainnya.                                                   |
| **NAME_PORTFOLIO**              | Jenis portofolio produk kredit, misalnya *POS*, *Cash*, *Cards*, atau lainnya.                                                                                |
| **NAME_PRODUCT_TYPE**           | Jenis produk kredit berdasarkan skema penawarannya, misalnya *walk-in*, *x-sell*, atau produk lainnya.                                                        |
| **CHANNEL_TYPE**                | Kanal atau saluran tempat aplikasi kredit diajukan, misalnya melalui cabang, dealer, agen, atau kanal lain.                                                   |
| **SELLERPLACE_AREA**            | Luas area atau ukuran lokasi penjual/merchant tempat kredit digunakan.                                                                                        |
| **NAME_SELLER_INDUSTRY**        | Jenis industri penjual atau merchant tempat kredit digunakan, misalnya elektronik, connectivity, furniture, dan lainnya.                                      |
| **CNT_PAYMENT**                 | Jumlah total cicilan yang direncanakan untuk kredit tersebut.                                                                                                 |
| **NAME_YIELD_GROUP**            | Kelompok yield atau tingkat profitabilitas kredit, misalnya *low*, *middle*, atau *high*.                                                                     |
| **PRODUCT_COMBINATION**         | Kombinasi produk kredit yang digunakan, misalnya POS, cash loan, card, atau kombinasi produk lainnya.                                                         |
| **DAYS_FIRST_DRAWING**          | Jumlah hari relatif terhadap aplikasi saat ini ketika pencairan dana pertama dilakukan. Nilai khusus seperti 365243 biasanya menunjukkan data tidak tersedia. |
| **DAYS_FIRST_DUE**              | Jumlah hari relatif terhadap aplikasi saat ini ketika pembayaran pertama jatuh tempo. Nilai negatif berarti terjadi di masa lalu.                             |
| **DAYS_LAST_DUE_1ST_VERSION**   | Jumlah hari relatif terhadap aplikasi saat ini untuk tanggal jatuh tempo terakhir berdasarkan jadwal kontrak awal.                                            |
| **DAYS_LAST_DUE**               | Jumlah hari relatif terhadap aplikasi saat ini untuk tanggal jatuh tempo terakhir yang sebenarnya terjadi.                                                    |
| **DAYS_TERMINATION**            | Jumlah hari relatif terhadap aplikasi saat ini ketika kontrak kredit sebelumnya benar-benar berakhir atau dihentikan.                                         |
| **NFLAG_INSURED_ON_APPROVAL**   | Indikator apakah kredit diasuransikan saat disetujui, dengan nilai 1 = ya dan 0 = tidak.                                                                      |


### 1.2.4 Bureau Balance

| **Variable**       | **Description**                                                                                                                     |
| :----------------- | :---------------------------------------------------------------------------------------------------------------------------------- |
| **SK_ID_BUREAU**   | Identifier unik untuk setiap kredit di bureau. Digunakan untuk menghubungkan ke tabel `bureau`.                                     |
| **MONTHS_BALANCE** | Jarak waktu dalam bulan relatif terhadap waktu aplikasi saat ini. Nilai 0 = bulan terakhir, nilai negatif = bulan-bulan sebelumnya. |
| **STATUS**         | Status kredit pada bulan tersebut. Kode ini menunjukkan kondisi pembayaran atau status kredit.                                      |


| **Status** | **Makna**                                         |
| ---------- | ------------------------------------------------- |
| **C**      | Closed (kredit sudah ditutup/lunas)               |
| **0**      | Tidak ada keterlambatan pembayaran                |
| **1**      | Terlambat 1–30 hari                               |
| **2**      | Terlambat 31–60 hari                              |
| **3**      | Terlambat 61–90 hari                              |
| **4**      | Terlambat 91–120 hari                             |
| **5**      | Terlambat lebih dari 120 hari                     |
| **X**      | Tidak ada data untuk bulan tersebut / tidak aktif |


### 1.2.5 POS Cash Balance


| **Variable**              | **Description**                                                                                                              |
| :------------------------ | :--------------------------------------------------------------------------------------------------------------------------- |
| **SK_ID_PREV**            | Identifier unik untuk setiap aplikasi kredit sebelumnya (menghubungkan ke `previous_application`).                           |
| **SK_ID_CURR**            | Identifier unik nasabah (client), digunakan untuk menghubungkan ke tabel utama (`application`).                              |
| **MONTHS_BALANCE**        | Jarak waktu dalam bulan relatif terhadap waktu aplikasi saat ini. Nilai 0 = bulan terbaru, nilai negatif = bulan sebelumnya. |
| **CNT_INSTALMENT**        | Jumlah total cicilan yang direncanakan dalam kontrak kredit.                                                                 |
| **CNT_INSTALMENT_FUTURE** | Jumlah cicilan yang masih tersisa (belum dibayar) pada bulan tersebut.                                                       |
| **NAME_CONTRACT_STATUS**  | Status kontrak kredit pada bulan tersebut (misalnya *Active*, *Completed*, dll).                                             |
| **SK_DPD**                | Days Past Due: jumlah hari keterlambatan pembayaran (termasuk semua keterlambatan).                                          |
| **SK_DPD_DEF**            | Days Past Due dengan threshold default: jumlah hari keterlambatan yang dianggap sebagai default (biasanya lebih serius).     |


### 1.2.6 Installments Payment

| **Variable**               | **Description**                                                                                                           |
| :------------------------- | :------------------------------------------------------------------------------------------------------------------------ |
| **SK_ID_PREV**             | Identifier unik untuk setiap aplikasi kredit sebelumnya (menghubungkan ke `previous_application`).                        |
| **SK_ID_CURR**             | Identifier unik nasabah (client), digunakan untuk menghubungkan ke tabel utama (`application`).                           |
| **NUM_INSTALMENT_VERSION** | Versi jadwal cicilan. Nilai >1 biasanya menunjukkan adanya perubahan jadwal pembayaran (misalnya restrukturisasi kredit). |
| **NUM_INSTALMENT_NUMBER**  | Nomor urutan cicilan dalam kontrak (misalnya cicilan ke-1, ke-2, dst).                                                    |
| **DAYS_INSTALMENT**        | Hari jatuh tempo cicilan (dalam format hari relatif, nilai negatif berarti di masa lalu sebelum aplikasi saat ini).       |
| **DAYS_ENTRY_PAYMENT**     | Hari aktual pembayaran dilakukan.                                                                                         |
| **AMT_INSTALMENT**         | Jumlah cicilan yang seharusnya dibayar.                                                                                   |
| **AMT_PAYMENT**            | Jumlah pembayaran yang benar-benar dilakukan oleh nasabah.                                                                |


### 1.2.7 Credit Card Balance

| **Variable**                   | **Description**                                                                                                   |
| :----------------------------- | :---------------------------------------------------------------------------------------------------------------- |
| **SK_ID_PREV**                 | Identifier unik untuk setiap akun kartu kredit (menghubungkan ke `previous_application`).                         |
| **SK_ID_CURR**                 | Identifier unik nasabah (client), digunakan untuk menghubungkan ke tabel utama (`application`).                   |
| **MONTHS_BALANCE**             | Jarak waktu dalam bulan relatif terhadap waktu aplikasi saat ini (0 = bulan terbaru, negatif = bulan sebelumnya). |
| **AMT_BALANCE**                | Total saldo kartu kredit pada bulan tersebut (jumlah yang terutang).                                              |
| **AMT_CREDIT_LIMIT_ACTUAL**    | Batas maksimum kredit kartu (credit limit).                                                                       |
| **AMT_DRAWINGS_ATM_CURRENT**   | Jumlah penarikan tunai melalui ATM pada bulan tersebut.                                                           |
| **AMT_DRAWINGS_CURRENT**       | Total penarikan dana (semua jenis transaksi) pada bulan tersebut.                                                 |
| **AMT_DRAWINGS_OTHER_CURRENT** | Jumlah penarikan selain ATM dan POS (misalnya transfer atau lainnya).                                             |
| **AMT_DRAWINGS_POS_CURRENT**   | Jumlah transaksi pembelian melalui POS (Point of Sale).                                                           |
| **AMT_INST_MIN_REGULARITY**    | Jumlah minimum pembayaran yang harus dibayar pada bulan tersebut.                                                 |
| **AMT_RECIVABLE**              | Jumlah piutang yang dapat ditagih pada bulan tersebut.                                                            |
| **AMT_TOTAL_RECEIVABLE**       | Total seluruh piutang nasabah terhadap kartu kredit.                                                              |
| **CNT_DRAWINGS_ATM_CURRENT**   | Jumlah transaksi penarikan ATM pada bulan tersebut.                                                               |
| **CNT_DRAWINGS_CURRENT**       | Total jumlah transaksi penarikan (semua jenis).                                                                   |
| **CNT_DRAWINGS_OTHER_CURRENT** | Jumlah transaksi penarikan lainnya.                                                                               |
| **CNT_DRAWINGS_POS_CURRENT**   | Jumlah transaksi pembelian melalui POS.                                                                           |
| **CNT_INSTALMENT_MATURE_CUM**  | Jumlah total cicilan akumulatif yang sudah jatuh tempo hingga saat itu.                                                      |
| **NAME_CONTRACT_STATUS**       | Status akun kartu kredit (misalnya *Active*, *Completed*, dll).                                                   |
| **SK_DPD**                     | Days Past Due: jumlah hari keterlambatan pembayaran.                                                              |
| **SK_DPD_DEF**                 | Days Past Due untuk default (keterlambatan serius).                                                               |


# **2.&nbsp;Data Preparation**

## 2.1 Application Train

In [ ]:
df_application.info()

In [ ]:
df_application.describe()

### 2.1.1 Reduce RAM Usage


In [ ]:
def reduce_mem_usage_safe(df_application, use_float16=False):
    """Mengurangi penggunaan memori dengan risiko presisi lebih rendah."""
    df_application = df_application.copy()

    start_mem = df_application.memory_usage(deep=True).sum() / 1024**2
    print(f"➜ Memory usage awal: {start_mem:.4f} MB")

    for col in df_application.columns:
        col_type = df_application[col].dtype

        if pd.api.types.is_bool_dtype(col_type):
            continue

        if pd.api.types.is_datetime64_any_dtype(col_type):
            continue

        if pd.api.types.is_integer_dtype(col_type):
            df_application[col] = pd.to_numeric(df_application[col], downcast="integer")

        elif pd.api.types.is_float_dtype(col_type):
            if use_float16:
                df_application[col] = pd.to_numeric(df_application[col], downcast="float")
            else:
                if (
                    df_application[col].min() >= np.finfo(np.float32).min
                    and df_application[col].max() <= np.finfo(np.float32).max
                ):
                    df_application[col] = df_application[col].astype(np.float32)

        elif pd.api.types.is_object_dtype(col_type):
            num_unique = df_application[col].nunique(dropna=False)
            num_total = len(df_application[col])

            if num_total > 0 and num_unique / num_total < 0.5:
                df_application[col] = df_application[col].astype("category")

    end_mem = df_application.memory_usage(deep=True).sum() / 1024**2
    print(f"➜ Memory usage setelah optimasi: {end_mem:.4f} MB")

    if start_mem > 0:
        print(f"➜ Berhasil menghemat memori sebesar: {100 * (start_mem - end_mem) / start_mem:.1f}%\n")

    return df_application

In [ ]:
print("--- TAHAP 1: OPTIMASI MEMORI ---")
df_application = reduce_mem_usage_safe(df_application)

### 2.1.2 Drop Column: Missing values >=60%

In [ ]:
def drop_high_missing_cols(df_application, threshold=0.60):
    """ Menghapus kolom yang memiliki missing value di atas threshold (default 60%) """
    missing_pct = df_application.isnull().mean()
    cols_to_drop = missing_pct[missing_pct > threshold].index

    print(f"➜ Menghapus {len(cols_to_drop)} kolom dengan missing value > {threshold*100}%")

    # --- BAGIAN BARU: Menampilkan nama kolom yang dihapus ---
    if len(cols_to_drop) > 0:
        print(f"➜ Daftar kolom yang dihapus:\n{list(cols_to_drop)}\n")
    # --------------------------------------------------------

    df_application.drop(columns=cols_to_drop, inplace=True)
    return df_application

print("--- TAHAP 2: CLEANING MISSING VALUES ---")
df_application = drop_high_missing_cols(df_application, threshold=0.60)

print(f"Dimensi data saat ini: {df_application.shape}")

### 2.1.3 Drop Column: Constant Value & Overlap Correlation


In [ ]:
import gc
import numpy as np
import pandas as pd

def drop_constant_features(df_application):
    """Menghapus kolom yang nilainya konstan / sama semua."""
    df_application = df_application.copy()

    constant_cols = [
        col for col in df_application.columns
        if df_application[col].nunique(dropna=False) <= 1
    ]

    print(f"➜ Menghapus {len(constant_cols)} kolom konstan")

    if len(constant_cols) > 0:
        print(f"   Daftar kolom konstan: {constant_cols}\n")

    df_application = df_application.drop(columns=constant_cols)

    return df_application, constant_cols


def drop_highly_correlated_features(df_application, threshold=0.95):
    """Menghapus fitur numerik yang memiliki korelasi sangat tinggi."""
    df_application = df_application.copy()

    numeric_df = df_application.select_dtypes(include=[np.number])
    corr_matrix = numeric_df.corr().abs()

    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    to_drop = []
    correlated_pairs = []

    for column in upper.columns:
        high_corr_rows = upper.index[upper[column] > threshold].tolist()

        if high_corr_rows:
            to_drop.append(column)

            for row in high_corr_rows:
                corr_value = upper.loc[row, column]
                correlated_pairs.append(
                    f"Kolom [{column}] dihapus karena mirip {corr_value*100:.1f}% dengan kolom [{row}]"
                )

    to_drop = list(set(to_drop))

    print(
        f"➜ Menghapus {len(to_drop)} kolom karena korelasi saling tumpang tindih (> {threshold*100:.0f}%)"
    )

    if len(correlated_pairs) > 0:
        print("   Alasan penghapusan:")
        for pair in correlated_pairs:
            print(f"   - {pair}")
        print()

    df_application = df_application.drop(columns=to_drop)

    del numeric_df, corr_matrix, upper
    gc.collect()

    return df_application, to_drop, correlated_pairs

Buat Feature sebelum drop


In [ ]:
# sebelum drop correlated
df_application["CREDIT_GOODS_RATIO"] = df_application["AMT_CREDIT"] / df_application["AMT_GOODS_PRICE"]
df_application["CREDIT_INCOME_RATIO"] = df_application["AMT_CREDIT"] / df_application["AMT_INCOME_TOTAL"]
df_application["ANNUITY_INCOME_RATIO"] = df_application["AMT_ANNUITY"] / df_application["AMT_INCOME_TOTAL"]
df_application["DAYS_EMPLOYED_ANOM"] = df_application["DAYS_EMPLOYED"] == 365243
df_application["DAYS_EMPLOYED"] = df_application["DAYS_EMPLOYED"].replace(365243, np.nan)

In [ ]:
print("--- TAHAP 3: FEATURE SELECTION ---")

df_application, constant_cols = drop_constant_features(df_application)

df_application, high_corr_cols, correlated_pairs = drop_highly_correlated_features(
    df_application,
    threshold=0.95
)

print(f"Dimensi data saat ini: {df_application.shape}")

### 2.1.4 Agregate Column = FLAG DOCUMENT

In [ ]:
def aggregate_document_flags(df_application):
    """Menggabungkan kolom FLAG_DOCUMENT_* menjadi TOTAL_DOCUMENT_FLAGS."""
    df_application = df_application.copy()

    doc_flag_cols = [
        col for col in df_application.columns
        if col.startswith("FLAG_DOCUMENT_")
    ]

    if len(doc_flag_cols) > 0:
        for col in doc_flag_cols:
            df_application[col] = pd.to_numeric(
                df_application[col],
                errors="coerce"
            ).fillna(0)

        print(f"➜ Membuat fitur 'TOTAL_DOCUMENT_FLAGS' dari {len(doc_flag_cols)} kolom FLAG_DOCUMENT_")

        df_application["TOTAL_DOCUMENT_FLAGS"] = df_application[doc_flag_cols].sum(axis=1)

        df_application = df_application.drop(columns=doc_flag_cols)

        print(f"➜ Menghapus {len(doc_flag_cols)} kolom FLAG_DOCUMENT_ asli")
    else:
        print("➜ Tidak ada kolom FLAG_DOCUMENT_ yang ditemukan")

    return df_application

In [ ]:
print("--- TAHAP 4: AGREGASI FLAG DOCUMENT ---")

df_application = aggregate_document_flags(df_application)

print(f"Dimensi data saat ini: {df_application.shape}")

In [ ]:
df_application.info()

### 2.1.5 Drop Row: Small Missing Cols

In [ ]:
missing = df_application.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

In [ ]:
small_missing_cols = [
    "AMT_ANNUITY",
    "ANNUITY_INCOME_RATIO",
    "CNT_FAM_MEMBERS",
    "DAYS_LAST_PHONE_CHANGE"
]

df_application = df_application.dropna(subset=small_missing_cols)

print(df_application.shape)

In [ ]:
missing = df_application.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

### 2.1.6 Fill Na: SOCIAL_CIRCLE

In [ ]:
social_cols = [col for col in df_application.columns if "SOCIAL_CIRCLE" in col]

df_application[social_cols].isna().sum()

In [ ]:
print("Missing semua social_cols:", df_application[social_cols].isna().all(axis=1).sum())
print("Missing salah satu social_cols:", df_application[social_cols].isna().any(axis=1).sum())

In [ ]:
print("--- TAHAP 5: HANDLE SOCIAL_CIRCLE MISSING ---")

social_cols = [
    col for col in df_application.columns
    if "SOCIAL_CIRCLE" in col
]

if len(social_cols) > 0:
    print("Kolom social circle:", social_cols)

    print("Missing semua social_cols:", df_application[social_cols].isna().all(axis=1).sum())
    print("Missing salah satu social_cols:", df_application[social_cols].isna().any(axis=1).sum())

    df_application["SOCIAL_CIRCLE_MISSING"] = (
        df_application[social_cols].isna().all(axis=1).astype(int)
    )

    df_application[social_cols] = df_application[social_cols].fillna(0)

    print("Setelah imputasi social circle:")
    print(df_application[social_cols + ["SOCIAL_CIRCLE_MISSING"]].isna().sum())
else:
    print("Tidak ada kolom SOCIAL_CIRCLE ditemukan")

print("Dimensi data:", df_application.shape)

### 2.1.7 Fill NA: AMT_CREDIT_BUREAU

In [ ]:
bureau_cols = [
    col for col in df_application.columns
    if "AMT_REQ_CREDIT_BUREAU" in col
]

if len(bureau_cols) > 0:
    print("Kolom bureau:", bureau_cols)

    print("Missing semua bureau_cols:", df_application[bureau_cols].isna().all(axis=1).sum())
    print("Missing salah satu bureau_cols:", df_application[bureau_cols].isna().any(axis=1).sum())

    df_application["BUREAU_INFO_MISSING"] = (
        df_application[bureau_cols].isna().all(axis=1).astype(int)
    )

    df_application[bureau_cols] = df_application[bureau_cols].fillna(0)

    print("Setelah imputasi bureau:")
    print(df_application[bureau_cols + ["BUREAU_INFO_MISSING"]].isna().sum())
else:
    print("Tidak ada kolom AMT_REQ_CREDIT_BUREAU ditemukan")

print("Dimensi data:", df_application.shape)

### 2.1.8 Imputation: Category column with "Unknown"

In [ ]:
print("IMPUTASI KATEGORIKAL DENGAN UNKNOWN ---")

cat_cols = df_application.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Jumlah kolom kategorikal: {len(cat_cols)}")

for col in cat_cols:
    if str(df_application[col].dtype) == "category":
        if "Unknown" not in df_application[col].cat.categories:
            df_application[col] = df_application[col].cat.add_categories("Unknown")

    df_application[col] = df_application[col].fillna("Unknown")

print("Total missing kategorikal setelah imputasi:", df_application[cat_cols].isna().sum().sum())
print("Dimensi data:", df_application.shape)

### 2.1.9 Imputation: Numeric Column with Median

In [ ]:
print("--- IMPUTASI NUMERIK DENGAN MEDIAN ---")

num_cols = df_application.select_dtypes(include=[np.number]).columns.tolist()

print(f"Jumlah kolom numerik: {len(num_cols)}")

for col in num_cols:
    if df_application[col].isna().sum() > 0:
        median_value = df_application[col].median()
        df_application[col] = df_application[col].fillna(median_value)

print("Total missing numerik setelah imputasi:", df_application[num_cols].isna().sum().sum())
print("Dimensi data:", df_application.shape)

In [ ]:
print("--- FINAL CHECK ---")

total_missing = df_application.isna().sum().sum()
print("Total missing value tersisa:", total_missing)

missing_summary = (
    pd.DataFrame({
        "missing_count": df_application.isna().sum(),
        "missing_pct": df_application.isna().mean() * 100
    })
    .sort_values("missing_count", ascending=False)
)

print("\nKolom yang masih memiliki missing:")
print(missing_summary[missing_summary["missing_count"] > 0].to_string())

print("\nDimensi akhir:", df_application.shape)
print("=== CLEANING SELESAI ===")

In [ ]:
df_application.info()

In [ ]:
df_application["TOTAL_DOCUMENT_FLAGS"] = df_application["TOTAL_DOCUMENT_FLAGS"].astype("int8")
df_application["SOCIAL_CIRCLE_MISSING"] = df_application["SOCIAL_CIRCLE_MISSING"].astype("int8")
df_application["BUREAU_INFO_MISSING"] = df_application["BUREAU_INFO_MISSING"].astype("int8")
df_application["DAYS_EMPLOYED_ANOM"] = df_application["DAYS_EMPLOYED_ANOM"].astype("int8")

### 2.1.10 Detect Duplicate Row

In [ ]:
print("Duplicate rows:", df_application.duplicated().sum())
print("Duplicate SK_ID_CURR:", df_application['SK_ID_CURR'].duplicated().sum())
df_application = df_application.drop_duplicates()

In [ ]:
df_application[df_application["DAYS_EMPLOYED_ANOM"] == 1]["NAME_INCOME_TYPE"].value_counts(dropna=False)

Hari kerja kosong berarti Pensiunan dan Pengangguran

### 2.1.11 Capping: Income, Circle, Credit, Anak, dan Jumlah Anggota Keluarga

**Before**

In [ ]:
capped_cols = [
    'AMT_INCOME_TOTAL', 'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
    'DEF_60_CNT_SOCIAL_CIRCLE', 'AMT_REQ_CREDIT_BUREAU_YEAR',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS'
]


cols_to_plot = [col for col in capped_cols if col in df_application.columns]

n_cols = 3
n_rows = (len(cols_to_plot) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    sns.boxplot(x=df_application[col], ax=axes[i], color='skyblue')
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
cap_config = {
    "AMT_INCOME_TOTAL": 0.99,
    "OBS_30_CNT_SOCIAL_CIRCLE": 0.98,
    "DEF_30_CNT_SOCIAL_CIRCLE": 0.98,
    "DEF_60_CNT_SOCIAL_CIRCLE": 0.98,
    "AMT_REQ_CREDIT_BUREAU_YEAR": 0.99,
    "CNT_CHILDREN": 0.995,
    "CNT_FAM_MEMBERS": 0.995
}

for col, q in cap_config.items():
    if col in df_application.columns:
        cap_val = df_application[col].quantile(q)

        df_application[f"{col}_OUTLIER"] = (
            df_application[col] > cap_val
        ).astype("int8")

        df_application[col] = df_application[col].clip(upper=cap_val)

        print(f"{col}: capped at {q*100:.1f} percentile → {cap_val:.2f}")

In [ ]:
[col for col in df_application.columns if col.endswith("_OUTLIER")]

**After**

In [ ]:
cols_to_plot = [col for col in capped_cols if col in df_application.columns]

n_cols = 3
n_rows = (len(cols_to_plot) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    sns.boxplot(x=df_application[col], ax=axes[i], color='skyblue')
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
df_application.info()

## 2.2 Bureau

In [ ]:
print("--- CEK AWAL: BUREAU ---")
print("Dimensi awal:", df_bureau.shape)
print("\nInfo:")
df_bureau.info()

print("\nJumlah missing value:")
print(df_bureau.isna().sum().sort_values(ascending=False).to_string())

print("\nDuplicate rows:", df_bureau.duplicated().sum())

if "SK_ID_BUREAU" in df_bureau.columns:
    print("Duplicate SK_ID_BUREAU:", df_bureau["SK_ID_BUREAU"].duplicated().sum())

if "SK_ID_CURR" in df_bureau.columns:
    print("Jumlah unique SK_ID_CURR:", df_bureau["SK_ID_CURR"].nunique())

### 2.2.1 Reduce RAM Usage

In [ ]:
def reduce_mem_usage_safe(df_bureau, use_float16=False):
    """Mengurangi penggunaan memori dataframe secara aman."""
    df_bureau = df_bureau.copy()

    start_mem = df_bureau.memory_usage(deep=True).sum() / 1024**2
    print(f"➜ Memory usage awal: {start_mem:.4f} MB")

    for col in df_bureau.columns:
        col_type = df_bureau[col].dtype

        if pd.api.types.is_bool_dtype(col_type):
            continue

        if pd.api.types.is_datetime64_any_dtype(col_type):
            continue

        if pd.api.types.is_integer_dtype(col_type):
            df_bureau[col] = pd.to_numeric(df_bureau[col], downcast="integer")

        elif pd.api.types.is_float_dtype(col_type):
            if use_float16:
                df_bureau[col] = pd.to_numeric(df_bureau[col], downcast="float")
            else:
                if (
                    df_bureau[col].min() >= np.finfo(np.float32).min
                    and df_bureau[col].max() <= np.finfo(np.float32).max
                ):
                    df_bureau[col] = df_bureau[col].astype(np.float32)

        elif pd.api.types.is_object_dtype(col_type):
            num_unique = df_bureau[col].nunique(dropna=False)
            num_total = len(df_bureau[col])

            if num_total > 0 and num_unique / num_total < 0.5:
                df_bureau[col] = df_bureau[col].astype("category")

    end_mem = df_bureau.memory_usage(deep=True).sum() / 1024**2
    print(f"➜ Memory usage setelah optimasi: {end_mem:.4f} MB")

    if start_mem > 0:
        print(f"➜ Hemat memori: {100 * (start_mem - end_mem) / start_mem:.1f}%\n")

    return df_bureau

In [ ]:
print("--- OPTIMASI MEMORI BUREAU ---")
df_bureau = reduce_mem_usage_safe(df_bureau)

In [ ]:
df_bureau.info()

### 2.2.2 Duplicate Value

In [ ]:
print("Duplicate rows:", df_bureau.duplicated().sum())

if "SK_ID_BUREAU" in df_bureau.columns:
    print("Duplicate SK_ID_BUREAU:", df_bureau["SK_ID_BUREAU"].duplicated().sum())

if "SK_ID_CURR" in df_bureau.columns:
    print("Unique SK_ID_CURR:", df_bureau["SK_ID_CURR"].nunique())

### 2.2.3 Missing Value

In [ ]:
missing_bureau = (
    pd.DataFrame({
        "missing_count": df_bureau.isna().sum(),
        "missing_pct": df_bureau.isna().mean() * 100
    })
    .sort_values("missing_count", ascending=False)
)

print(missing_bureau.to_string())

In [ ]:
print("--- BUAT MISSING FLAGS ---")

missing_flag_cols = [
    "AMT_ANNUITY",
    "AMT_CREDIT_MAX_OVERDUE",
    "DAYS_ENDDATE_FACT",
    "AMT_CREDIT_SUM_LIMIT",
    "AMT_CREDIT_SUM_DEBT",
    "DAYS_CREDIT_ENDDATE",
    "AMT_CREDIT_SUM"
]

for col in missing_flag_cols:
    if col in df_bureau.columns:
        flag_col = f"{col}_MISSING"
        df_bureau[flag_col] = df_bureau[col].isna().astype("int8")
        print(f"{flag_col}: dibuat | jumlah missing awal = {df_bureau[flag_col].sum()}")

print("Dimensi setelah missing flags:", df_bureau.shape)

In [ ]:
[col for col in df_bureau.columns if col.endswith("_MISSING")]

Isi dengan 0

In [ ]:
print("--- IMPUTASI AMOUNT DENGAN 0 ---")

zero_fill_cols = [
    "AMT_CREDIT_MAX_OVERDUE",
    "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_LIMIT",
    "AMT_CREDIT_SUM_OVERDUE",
    "AMT_ANNUITY"
]

for col in zero_fill_cols:
    if col in df_bureau.columns:
        missing_before = df_bureau[col].isna().sum()
        df_bureau[col] = df_bureau[col].fillna(0)
        missing_after = df_bureau[col].isna().sum()

        print(f"{col}: missing sebelum={missing_before}, sesudah={missing_after}")

Isi dengan Median

In [ ]:
print("--- IMPUTASI AMT_CREDIT_SUM ---")

if "AMT_CREDIT_SUM" in df_bureau.columns:
    missing_before = df_bureau["AMT_CREDIT_SUM"].isna().sum()
    median_credit_sum = df_bureau["AMT_CREDIT_SUM"].median()

    df_bureau["AMT_CREDIT_SUM"] = df_bureau["AMT_CREDIT_SUM"].fillna(median_credit_sum)

    print("Missing sebelum:", missing_before)
    print("Median AMT_CREDIT_SUM:", median_credit_sum)
    print("Missing sesudah:", df_bureau["AMT_CREDIT_SUM"].isna().sum())

In [ ]:
print("--- IMPUTASI KOLOM DAYS KHUSUS ---")

special_days_cols = [
    "DAYS_ENDDATE_FACT",
    "DAYS_CREDIT_ENDDATE"
]

for col in special_days_cols:
    if col in df_bureau.columns:
        missing_before = df_bureau[col].isna().sum()
        median_val = df_bureau[col].median()

        df_bureau[col] = df_bureau[col].fillna(median_val)

        print(f"{col}:")
        print(f"  missing sebelum = {missing_before}")
        print(f"  median          = {median_val}")
        print(f"  missing sesudah = {df_bureau[col].isna().sum()}")

### 2.2.4 Anomali Days Value

In [ ]:
print("--- CEK ANOMALI 365243 PADA DAYS_* ---")

days_cols = [col for col in df_bureau.columns if col.startswith("DAYS_")]

for col in days_cols:
    anom_count = (df_bureau[col] == 365243).sum()
    print(f"{col}: {anom_count}")

### 2.2.5 Category type Handle

In [ ]:
print("--- HANDLE KATEGORIKAL ---")

cat_cols = df_bureau.select_dtypes(include=["object", "category"]).columns.tolist()

print("Kolom kategorikal:", cat_cols)

for col in cat_cols:
    if str(df_bureau[col].dtype) == "category":
        if "Unknown" not in df_bureau[col].cat.categories:
            df_bureau[col] = df_bureau[col].cat.add_categories("Unknown")

    df_bureau[col] = df_bureau[col].fillna("Unknown")

print("Missing kategorikal setelah imputasi:")
print(df_bureau[cat_cols].isna().sum())

In [ ]:
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df_bureau[col].value_counts(dropna=False))

### 2.2.6 Simple Feature Engineering

In [ ]:
df_bureau.info()

In [ ]:
print("--- FEATURE SEDERHANA BUREAU RECORD ---")

if {"AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM"}.issubset(df_bureau.columns):
    df_bureau["BUREAU_DEBT_CREDIT_RATIO"] = (
        df_bureau["AMT_CREDIT_SUM_DEBT"] /
        (df_bureau["AMT_CREDIT_SUM"] + 1)
    )
    print("BUREAU_DEBT_CREDIT_RATIO dibuat")

if {"AMT_CREDIT_SUM_OVERDUE", "AMT_CREDIT_SUM"}.issubset(df_bureau.columns):
    df_bureau["BUREAU_OVERDUE_CREDIT_RATIO"] = (
        df_bureau["AMT_CREDIT_SUM_OVERDUE"] /
        (df_bureau["AMT_CREDIT_SUM"] + 1)
    )
    print("BUREAU_OVERDUE_CREDIT_RATIO dibuat")

if {"AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM"}.issubset(df_bureau.columns):
    df_bureau["BUREAU_LIMIT_CREDIT_RATIO"] = (
        df_bureau["AMT_CREDIT_SUM_LIMIT"] /
        (df_bureau["AMT_CREDIT_SUM"] + 1)
    )
    print("BUREAU_LIMIT_CREDIT_RATIO dibuat")

if {"AMT_CREDIT_SUM_OVERDUE", "AMT_CREDIT_SUM_DEBT"}.issubset(df_bureau.columns):
    df_bureau["BUREAU_OVERDUE_DEBT_RATIO"] = (
        df_bureau["AMT_CREDIT_SUM_OVERDUE"] /
        (df_bureau["AMT_CREDIT_SUM_DEBT"] + 1)
    )
    print("BUREAU_OVERDUE_DEBT_RATIO dibuat")

if {"DAYS_CREDIT_ENDDATE", "DAYS_CREDIT"}.issubset(df_bureau.columns):
    df_bureau["BUREAU_CREDIT_DURATION"] = (
        df_bureau["DAYS_CREDIT_ENDDATE"] -
        df_bureau["DAYS_CREDIT"]
    )
    print("BUREAU_CREDIT_DURATION dibuat")

if {"DAYS_CREDIT_UPDATE", "DAYS_CREDIT"}.issubset(df_bureau.columns):
    df_bureau["BUREAU_UPDATE_RECENCY"] = (
        df_bureau["DAYS_CREDIT_UPDATE"] -
        df_bureau["DAYS_CREDIT"]
    )
    print("BUREAU_UPDATE_RECENCY dibuat")

print("Dimensi setelah fitur bureau:", df_bureau.shape)

### 2.2.7 Deep Cleaning Remaining Value

In [ ]:
print("--- : CLEAN INF DAN MISSING TERSISA ---")

df_bureau = df_bureau.replace([np.inf, -np.inf], 0)

num_cols = df_bureau.select_dtypes(include=[np.number]).columns.tolist()

for col in num_cols:
    missing_before = df_bureau[col].isna().sum()

    if missing_before > 0:
        median_val = df_bureau[col].median()
        df_bureau[col] = df_bureau[col].fillna(median_val)
        print(f"{col}: missing {missing_before} diisi median {median_val}")

print("Total missing:", df_bureau.isna().sum().sum())

In [ ]:
print("--- FINAL CHECK BUREAU CLEAN ---")

print("Dimensi akhir:", df_bureau.shape)
print("Total missing:", df_bureau.isna().sum().sum())
print("Duplicate rows:", df_bureau.duplicated().sum())

if "SK_ID_BUREAU" in df_bureau.columns:
    print("Duplicate SK_ID_BUREAU:", df_bureau["SK_ID_BUREAU"].duplicated().sum())

print("\nMissing summary:")
print(df_bureau.isna().sum().sort_values(ascending=False).to_string())

print("\nDtypes:")
print(df_bureau.dtypes)

### 2.2.8 Aggregation

#### 2.2.8.1 Basic Count

In [ ]:
print("--- CEK AWAL SEBELUM AGREGASI BUREAU ---")

print("Dimensi df_bureau_clean:", df_bureau.shape)
print("Jumlah unique SK_ID_CURR:", df_bureau["SK_ID_CURR"].nunique())
print("Jumlah unique SK_ID_BUREAU:", df_bureau["SK_ID_BUREAU"].nunique())

In [ ]:
print("--- TAHAP 1: BASIC BUREAU COUNT ---")

bureau_base_agg = df_bureau.groupby("SK_ID_CURR").agg(
    BUREAU_RECORD_COUNT=("SK_ID_BUREAU", "count"),
    BUREAU_UNIQUE_CREDIT_TYPE_COUNT=("CREDIT_TYPE", "nunique"),
    BUREAU_UNIQUE_CREDIT_ACTIVE_COUNT=("CREDIT_ACTIVE", "nunique"),
    BUREAU_UNIQUE_CURRENCY_COUNT=("CREDIT_CURRENCY", "nunique")
).reset_index()

print("Dimensi bureau_base_agg:", bureau_base_agg.shape)
bureau_base_agg.head()

#### 2.2.8.2 Numeric Aggregation

In [ ]:
print("--- TAHAP 2: NUMERIC AGGREGATION ---")

num_agg_cols = [
    "DAYS_CREDIT",
    "CREDIT_DAY_OVERDUE",
    "DAYS_CREDIT_ENDDATE",
    "DAYS_ENDDATE_FACT",
    "AMT_CREDIT_MAX_OVERDUE",
    "CNT_CREDIT_PROLONG",
    "AMT_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_LIMIT",
    "AMT_CREDIT_SUM_OVERDUE",
    "DAYS_CREDIT_UPDATE",
    "AMT_ANNUITY",
    "BUREAU_DEBT_CREDIT_RATIO",
    "BUREAU_OVERDUE_CREDIT_RATIO",
    "BUREAU_LIMIT_CREDIT_RATIO",
    "BUREAU_OVERDUE_DEBT_RATIO",
    "BUREAU_CREDIT_DURATION",
    "BUREAU_UPDATE_RECENCY"
]

num_agg_cols = [col for col in num_agg_cols if col in df_bureau.columns]

bureau_num_agg = df_bureau.groupby("SK_ID_CURR")[num_agg_cols].agg(
    ["mean", "max", "min", "sum"]
)

bureau_num_agg.columns = [
    "BUREAU_" + col.upper() + "_" + stat.upper()
    for col, stat in bureau_num_agg.columns
]

bureau_num_agg = bureau_num_agg.reset_index()

print("Dimensi bureau_num_agg:", bureau_num_agg.shape)
bureau_num_agg.head()

#### 2.2.8.3 Missing Flag Aggregation

In [ ]:
print("--- MISSING FLAG AGGREGATION ---")

missing_flag_cols = [
    col for col in df_bureau.columns
    if col.endswith("_MISSING")
]

bureau_missing_agg = df_bureau.groupby("SK_ID_CURR")[missing_flag_cols].agg(
    ["sum", "mean"]
)

bureau_missing_agg.columns = [
    "BUREAU_" + col.upper() + "_" + stat.upper()
    for col, stat in bureau_missing_agg.columns
]

bureau_missing_agg = bureau_missing_agg.reset_index()

print("Missing flag cols:", missing_flag_cols)
print("Dimensi bureau_missing_agg:", bureau_missing_agg.shape)
bureau_missing_agg.head()

#### 2.2.8.4 Active Credit Aggregation

In [ ]:
print("--- CREDIT_ACTIVE AGGREGATION ---")

bureau_active_dummies = pd.get_dummies(
    df_bureau[["SK_ID_CURR", "CREDIT_ACTIVE"]],
    columns=["CREDIT_ACTIVE"],
    prefix="BUREAU_CREDIT_ACTIVE"
)

bureau_active_agg = bureau_active_dummies.groupby("SK_ID_CURR").sum().reset_index()

print("Dimensi bureau_active_agg:", bureau_active_agg.shape)
bureau_active_agg.head()

#### 2.2.8.5 Credit Type Aggregation

In [ ]:
print("--- CREDIT_TYPE AGGREGATION ---")

bureau_type_dummies = pd.get_dummies(
    df_bureau[["SK_ID_CURR", "CREDIT_TYPE"]],
    columns=["CREDIT_TYPE"],
    prefix="BUREAU_CREDIT_TYPE"
)

bureau_type_agg = bureau_type_dummies.groupby("SK_ID_CURR").sum().reset_index()

print("Dimensi bureau_type_agg:", bureau_type_agg.shape)
bureau_type_agg.head()

#### 2.2.8.6 Credit Currency Aggregation

In [ ]:
print("--- CREDIT_CURRENCY AGGREGATION ---")

bureau_currency_dummies = pd.get_dummies(
    df_bureau[["SK_ID_CURR", "CREDIT_CURRENCY"]],
    columns=["CREDIT_CURRENCY"],
    prefix="BUREAU_CREDIT_CURRENCY"
)

bureau_currency_agg = bureau_currency_dummies.groupby("SK_ID_CURR").sum().reset_index()

print("Dimensi bureau_currency_agg:", bureau_currency_agg.shape)
bureau_currency_agg.head()

#### 2.2.8.7 Active Credit Aggregation

In [ ]:
print("--- ACTIVE CREDIT AGGREGATION ---")

bureau_active_only = df_bureau[
    df_bureau["CREDIT_ACTIVE"] == "Active"
].copy()

print("Jumlah record Active:", bureau_active_only.shape[0])

active_num_cols = [
    "AMT_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_LIMIT",
    "AMT_CREDIT_SUM_OVERDUE",
    "AMT_ANNUITY",
    "CREDIT_DAY_OVERDUE",
    "DAYS_CREDIT",
    "DAYS_CREDIT_ENDDATE",
    "BUREAU_DEBT_CREDIT_RATIO",
    "BUREAU_OVERDUE_CREDIT_RATIO"
]

active_num_cols = [
    col for col in active_num_cols
    if col in bureau_active_only.columns
]

bureau_active_num_agg = bureau_active_only.groupby("SK_ID_CURR")[active_num_cols].agg(
    ["mean", "max", "sum"]
)

bureau_active_num_agg.columns = [
    "BUREAU_ACTIVE_" + col.upper() + "_" + stat.upper()
    for col, stat in bureau_active_num_agg.columns
]

bureau_active_num_agg = bureau_active_num_agg.reset_index()

print("Dimensi bureau_active_num_agg:", bureau_active_num_agg.shape)
bureau_active_num_agg.head()

#### 2.2.8.8 Closed Credit Aggregation

In [ ]:
print("--- CLOSED CREDIT AGGREGATION ---")

bureau_closed_only = df_bureau[
    df_bureau["CREDIT_ACTIVE"] == "Closed"
].copy()

print("Jumlah record Closed:", bureau_closed_only.shape[0])

closed_num_cols = [
    "AMT_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_OVERDUE",
    "AMT_CREDIT_MAX_OVERDUE",
    "DAYS_CREDIT",
    "DAYS_ENDDATE_FACT",
    "BUREAU_CREDIT_DURATION"
]

closed_num_cols = [
    col for col in closed_num_cols
    if col in bureau_closed_only.columns
]

bureau_closed_num_agg = bureau_closed_only.groupby("SK_ID_CURR")[closed_num_cols].agg(
    ["mean", "max", "sum"]
)

bureau_closed_num_agg.columns = [
    "BUREAU_CLOSED_" + col.upper() + "_" + stat.upper()
    for col, stat in bureau_closed_num_agg.columns
]

bureau_closed_num_agg = bureau_closed_num_agg.reset_index()

print("Dimensi bureau_closed_num_agg:", bureau_closed_num_agg.shape)
bureau_closed_num_agg.head()

#### 2.2.8.9 Merge Result

In [ ]:
print("--- MERGE SEMUA AGGREGATION BUREAU ---")

bureau_aggs = [
    bureau_base_agg,
    bureau_num_agg,
    bureau_missing_agg,
    bureau_active_agg,
    bureau_type_agg,
    bureau_currency_agg,
    bureau_active_num_agg,
    bureau_closed_num_agg
]

df_bureau_agg = bureau_aggs[0]

for agg_df in bureau_aggs[1:]:
    df_bureau_agg = df_bureau_agg.merge(
        agg_df,
        on="SK_ID_CURR",
        how="left"
    )

df_bureau_agg = df_bureau_agg.fillna(0)

print("Dimensi df_bureau_agg:", df_bureau_agg.shape)
print("Unique SK_ID_CURR:", df_bureau_agg["SK_ID_CURR"].nunique())
print("Duplicate SK_ID_CURR:", df_bureau_agg["SK_ID_CURR"].duplicated().sum())
print("Total missing:", df_bureau_agg.isna().sum().sum())

df_bureau_agg.head()

### 2.2.9 Post Aggregation: Feature Engineering

In [ ]:
print("--- POST-AGGREGATION FEATURES ---")

if {
    "BUREAU_AMT_CREDIT_SUM_DEBT_SUM",
    "BUREAU_AMT_CREDIT_SUM_SUM"
}.issubset(df_bureau_agg.columns):
    df_bureau_agg["BUREAU_TOTAL_DEBT_CREDIT_RATIO"] = (
        df_bureau_agg["BUREAU_AMT_CREDIT_SUM_DEBT_SUM"] /
        (df_bureau_agg["BUREAU_AMT_CREDIT_SUM_SUM"] + 1)
    )

if {
    "BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM",
    "BUREAU_AMT_CREDIT_SUM_SUM"
}.issubset(df_bureau_agg.columns):
    df_bureau_agg["BUREAU_TOTAL_OVERDUE_CREDIT_RATIO"] = (
        df_bureau_agg["BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM"] /
        (df_bureau_agg["BUREAU_AMT_CREDIT_SUM_SUM"] + 1)
    )

if {
    "BUREAU_CREDIT_ACTIVE_Active",
    "BUREAU_RECORD_COUNT"
}.issubset(df_bureau_agg.columns):
    df_bureau_agg["BUREAU_ACTIVE_CREDIT_RATIO"] = (
        df_bureau_agg["BUREAU_CREDIT_ACTIVE_Active"] /
        (df_bureau_agg["BUREAU_RECORD_COUNT"] + 1)
    )

if {
    "BUREAU_CREDIT_ACTIVE_Closed",
    "BUREAU_RECORD_COUNT"
}.issubset(df_bureau_agg.columns):
    df_bureau_agg["BUREAU_CLOSED_CREDIT_RATIO"] = (
        df_bureau_agg["BUREAU_CREDIT_ACTIVE_Closed"] /
        (df_bureau_agg["BUREAU_RECORD_COUNT"] + 1)
    )

if {
    "BUREAU_CREDIT_ACTIVE_Bad debt",
    "BUREAU_RECORD_COUNT"
}.issubset(df_bureau_agg.columns):
    df_bureau_agg["BUREAU_BAD_DEBT_RATIO"] = (
        df_bureau_agg["BUREAU_CREDIT_ACTIVE_Bad debt"] /
        (df_bureau_agg["BUREAU_RECORD_COUNT"] + 1)
    )

df_bureau_agg = df_bureau_agg.replace([np.inf, -np.inf], 0).fillna(0)

print("Dimensi setelah post-agg features:", df_bureau_agg.shape)
print("Total missing:", df_bureau_agg.isna().sum().sum())

df_bureau_agg.head()

### 2.2.10 Reduce Memory Part II

In [ ]:
print("--- TAHAP 11: OPTIMASI MEMORI BUREAU AGG ---")

df_bureau_agg = reduce_mem_usage_safe(df_bureau_agg)

df_bureau_agg.info()

In [ ]:
print("--- FINAL CHECK BUREAU AGG ---")

print("Dimensi df_bureau_agg:", df_bureau_agg.shape)
print("Unique SK_ID_CURR:", df_bureau_agg["SK_ID_CURR"].nunique())
print("Duplicate SK_ID_CURR:", df_bureau_agg["SK_ID_CURR"].duplicated().sum())
print("Total missing:", df_bureau_agg.isna().sum().sum())

print("\nContoh 5 baris:")
display(df_bureau_agg.head())

print("\nKolom hasil agregasi:")
print(df_bureau_agg.columns.tolist())

### 2.2.11 Merge to Application Data

In [ ]:
print("--- MERGE BUREAU AGG KE APPLICATION ---")

print("Dimensi df_application sebelum merge:", df_application.shape)
print("Dimensi df_bureau_agg:", df_bureau_agg.shape)

df_application_full = df_application.merge(
    df_bureau_agg,
    on="SK_ID_CURR",
    how="left"
)

bureau_agg_cols = [
    col for col in df_bureau_agg.columns
    if col != "SK_ID_CURR"
]

df_application_full[bureau_agg_cols] = df_application_full[bureau_agg_cols].fillna(0)

print("Dimensi df_application setelah merge:", df_application_full.shape)
print("Total missing setelah merge:", df_application_full.isna().sum().sum())
print("Duplicate SK_ID_CURR setelah merge:", df_application_full["SK_ID_CURR"].duplicated().sum())

In [ ]:
print("Shape:", df_application_full.shape)
print("Missing:", df_application_full.isna().sum().sum())
print("Duplicate SK_ID_CURR:", df_application_full["SK_ID_CURR"].duplicated().sum())

## 2.3 Previous Application

In [ ]:
print("--- CEK AWAL: PREVIOUS APPLICATION ---")

print("Dimensi awal:", df_previousapp.shape)

print("\nInfo:")
df_previousapp.info()

print("\nMissing value:")
missing_previous = (
    pd.DataFrame({
        "missing_count": df_previousapp.isna().sum(),
        "missing_pct": df_previousapp.isna().mean() * 100
    })
    .sort_values("missing_count", ascending=False)
)

print(missing_previous.to_string())

print("\nDuplicate rows:", df_previousapp.duplicated().sum())

if "SK_ID_PREV" in df_previousapp.columns:
    print("Duplicate SK_ID_PREV:", df_previousapp["SK_ID_PREV"].duplicated().sum())

if "SK_ID_CURR" in df_previousapp.columns:
    print("Unique SK_ID_CURR:", df_previousapp["SK_ID_CURR"].nunique())

#### 2.3.1 Drop Column Almost 100% Missing

In [ ]:
print("--- DROP KOLOM MISSING TERLALU TINGGI ---")

cols_to_drop_prev = [
    "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED"
]

cols_to_drop_prev = [
    col for col in cols_to_drop_prev
    if col in df_previousapp.columns
]

print("Kolom yang akan dihapus:", cols_to_drop_prev)

df_previousapp = df_previousapp.drop(columns=cols_to_drop_prev)

print("Dimensi setelah drop:", df_previousapp.shape)

### 2.3.2 Anomaly Check & Handle

In [ ]:
print("--- CEK ANOMALI 365243 PADA DAYS_* ---")

days_cols = [
    col for col in df_previousapp.columns
    if col.startswith("DAYS_")
]

for col in days_cols:
    anom_count = (df_previousapp[col] == 365243).sum()
    print(f"{col}: {anom_count}")

In [ ]:
print("--- HANDLE ANOMALI 365243 ---")

days_cols = [
    col for col in df_previousapp.columns
    if col.startswith("DAYS_")
]

for col in days_cols:
    anom_count = (df_previousapp[col] == 365243).sum()

    if anom_count > 0:
        anom_col = f"{col}_ANOM"

        df_previousapp[anom_col] = (
            df_previousapp[col] == 365243
        ).astype("int8")

        df_previousapp[col] = df_previousapp[col].replace(365243, np.nan)

        print(f"{col}: {anom_count} nilai 365243 diganti NaN, flag {anom_col} dibuat")

print("Dimensi setelah handle anomali:", df_previousapp.shape)

### 2.3.3 Missing Flag

In [ ]:
print("--- UAT MISSING FLAGS ---")

important_missing_cols = [
    "AMT_ANNUITY",
    "AMT_CREDIT",
    "AMT_GOODS_PRICE",
    "AMT_DOWN_PAYMENT",
    "RATE_DOWN_PAYMENT",
    "NAME_TYPE_SUITE",
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
    "NFLAG_INSURED_ON_APPROVAL",
    "CNT_PAYMENT",
    "PRODUCT_COMBINATION"
]

important_missing_cols = [
    col for col in important_missing_cols
    if col in df_previousapp.columns
]

for col in important_missing_cols:
    flag_col = f"{col}_MISSING"
    df_previousapp[flag_col] = df_previousapp[col].isna().astype("int8")
    print(f"{flag_col}: dibuat | jumlah missing awal = {df_previousapp[flag_col].sum()}")

print("Dimensi setelah missing flags:", df_previousapp.shape)

### 2.3.4 Imputasi Amount

Strategi:

AMT_CREDIT, AMT_ANNUITY, AMT_GOODS_PRICE: Median.

AMT_DOWN_PAYMENT, RATE_DOWN_PAYMENT: 0, karena missing besar dan sering berarti tidak ada down payment / tidak tercatat.

In [ ]:
print("--- IMPUTASI AMOUNT ---")

median_fill_cols = [
    "AMT_ANNUITY",
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_GOODS_PRICE"
]

for col in median_fill_cols:
    if col in df_previousapp.columns:
        missing_before = df_previousapp[col].isna().sum()
        median_val = df_previousapp[col].median()

        df_previousapp[col] = df_previousapp[col].fillna(median_val)

        print(
            f"{col}: missing sebelum={missing_before}, "
            f"median={median_val:.2f}, "
            f"missing sesudah={df_previousapp[col].isna().sum()}"
        )

zero_fill_cols = [
    "AMT_DOWN_PAYMENT",
    "RATE_DOWN_PAYMENT"
]

for col in zero_fill_cols:
    if col in df_previousapp.columns:
        missing_before = df_previousapp[col].isna().sum()

        df_previousapp[col] = df_previousapp[col].fillna(0)

        print(
            f"{col}: missing sebelum={missing_before}, "
            f"diisi 0, "
            f"missing sesudah={df_previousapp[col].isna().sum()}"
        )

### 2.3.5 Imputasi Hari dengan Median

In [ ]:
print("--- IMPUTASI DAYS_* DENGAN MEDIAN ---")

days_cols = [
    col for col in df_previousapp.columns
    if col.startswith("DAYS_")
    and not col.endswith("_MISSING")
    and not col.endswith("_ANOM")
]

for col in days_cols:
    missing_before = df_previousapp[col].isna().sum()

    if missing_before > 0:
        median_val = df_previousapp[col].median()
        df_previousapp[col] = df_previousapp[col].fillna(median_val)

        print(
            f"{col}: missing sebelum={missing_before}, "
            f"median={median_val:.2f}, "
            f"missing sesudah={df_previousapp[col].isna().sum()}"
        )

### 2.3.6 CNT_PAYMENT & NFLAG_INSURED_ON_APPROVAL

In [ ]:
print("--- IMPUTASI CNT_PAYMENT DAN INSURED FLAG ---")

if "CNT_PAYMENT" in df_previousapp.columns:
    missing_before = df_previousapp["CNT_PAYMENT"].isna().sum()
    median_val = df_previousapp["CNT_PAYMENT"].median()

    df_previousapp["CNT_PAYMENT"] = df_previousapp["CNT_PAYMENT"].fillna(median_val)

    print(
        f"CNT_PAYMENT: missing sebelum={missing_before}, "
        f"median={median_val:.2f}, "
        f"missing sesudah={df_previousapp['CNT_PAYMENT'].isna().sum()}"
    )

if "NFLAG_INSURED_ON_APPROVAL" in df_previousapp.columns:
    missing_before = df_previousapp["NFLAG_INSURED_ON_APPROVAL"].isna().sum()

    df_previousapp["NFLAG_INSURED_ON_APPROVAL"] = (
        df_previousapp["NFLAG_INSURED_ON_APPROVAL"].fillna(0)
    )

    print(
        f"NFLAG_INSURED_ON_APPROVAL: missing sebelum={missing_before}, "
        f"diisi 0, "
        f"missing sesudah={df_previousapp['NFLAG_INSURED_ON_APPROVAL'].isna().sum()}"
    )

### 2.3.7 Imputasi Kategorikal

In [ ]:
print("--- HANDLE KATEGORIKAL ---")

cat_cols = df_previousapp.select_dtypes(include=["object", "category"]).columns.tolist()

print("Jumlah kolom kategorikal:", len(cat_cols))
print("Kolom kategorikal:", cat_cols)

for col in cat_cols:
    if str(df_previousapp[col].dtype) == "category":
        if "Unknown" not in df_previousapp[col].cat.categories:
            df_previousapp[col] = df_previousapp[col].cat.add_categories("Unknown")

    df_previousapp[col] = df_previousapp[col].fillna("Unknown")

print("Missing kategorikal setelah imputasi:")
print(df_previousapp[cat_cols].isna().sum().to_string())

In [ ]:
df_previousapp.info()

### 2.3.8 Reduce Memory

In [ ]:
print("--- OPTIMASI MEMORI PREVIOUS APPLICATION ---")

df_previousapp = reduce_mem_usage_safe(df_previousapp)

df_previousapp.info()

### 2.3.9 Simple Feature Engineering

In [ ]:
print("--- FEATURE SEDERHANA PREVIOUS APPLICATION ---")

if {"AMT_CREDIT", "AMT_APPLICATION"}.issubset(df_previousapp.columns):
    df_previousapp["PREV_CREDIT_APPLICATION_DIFF"] = (
        df_previousapp["AMT_CREDIT"] - df_previousapp["AMT_APPLICATION"]
    )

    df_previousapp["PREV_CREDIT_APPLICATION_RATIO"] = (
        df_previousapp["AMT_CREDIT"] / (df_previousapp["AMT_APPLICATION"] + 1)
    )

if {"AMT_DOWN_PAYMENT", "AMT_APPLICATION"}.issubset(df_previousapp.columns):
    df_previousapp["PREV_DOWN_PAYMENT_APPLICATION_RATIO"] = (
        df_previousapp["AMT_DOWN_PAYMENT"] / (df_previousapp["AMT_APPLICATION"] + 1)
    )

if {"AMT_ANNUITY", "AMT_CREDIT"}.issubset(df_previousapp.columns):
    df_previousapp["PREV_ANNUITY_CREDIT_RATIO"] = (
        df_previousapp["AMT_ANNUITY"] / (df_previousapp["AMT_CREDIT"] + 1)
    )

if {"AMT_CREDIT", "AMT_ANNUITY"}.issubset(df_previousapp.columns):
    df_previousapp["PREV_CREDIT_ANNUITY_RATIO"] = (
        df_previousapp["AMT_CREDIT"] / (df_previousapp["AMT_ANNUITY"] + 1)
    )

if {"AMT_GOODS_PRICE", "AMT_CREDIT"}.issubset(df_previousapp.columns):
    df_previousapp["PREV_GOODS_CREDIT_RATIO"] = (
        df_previousapp["AMT_GOODS_PRICE"] / (df_previousapp["AMT_CREDIT"] + 1)
    )

if {"DAYS_LAST_DUE_1ST_VERSION", "DAYS_FIRST_DUE"}.issubset(df_previousapp.columns):
    df_previousapp["PREV_PLANNED_DURATION"] = (
        df_previousapp["DAYS_LAST_DUE_1ST_VERSION"] -
        df_previousapp["DAYS_FIRST_DUE"]
    )

if {"DAYS_LAST_DUE", "DAYS_FIRST_DUE"}.issubset(df_previousapp.columns):
    df_previousapp["PREV_ACTUAL_DURATION"] = (
        df_previousapp["DAYS_LAST_DUE"] -
        df_previousapp["DAYS_FIRST_DUE"]
    )

if {"PREV_ACTUAL_DURATION", "PREV_PLANNED_DURATION"}.issubset(df_previousapp.columns):
    df_previousapp["PREV_DURATION_DIFF"] = (
        df_previousapp["PREV_ACTUAL_DURATION"] -
        df_previousapp["PREV_PLANNED_DURATION"]
    )

if "NAME_CONTRACT_STATUS" in df_previousapp.columns:
    df_previousapp["PREV_IS_APPROVED"] = (
        df_previousapp["NAME_CONTRACT_STATUS"] == "Approved"
    ).astype("int8")

    df_previousapp["PREV_IS_REFUSED"] = (
        df_previousapp["NAME_CONTRACT_STATUS"] == "Refused"
    ).astype("int8")

    df_previousapp["PREV_IS_CANCELED"] = (
        df_previousapp["NAME_CONTRACT_STATUS"] == "Canceled"
    ).astype("int8")

    df_previousapp["PREV_IS_UNUSED_OFFER"] = (
        df_previousapp["NAME_CONTRACT_STATUS"] == "Unused offer"
    ).astype("int8")

print("Dimensi setelah feature sederhana:", df_previousapp.shape)
print("Total missing:", df_previousapp.isna().sum().sum())

In [ ]:
num_cols = df_previousapp.select_dtypes(include=[np.number]).columns.tolist()

df_previousapp[num_cols] = df_previousapp[num_cols].replace([np.inf, -np.inf], 0)

for col in num_cols:
    if df_previousapp[col].isna().sum() > 0:
        df_previousapp[col] = df_previousapp[col].fillna(df_previousapp[col].median())

print("Total missing:", df_previousapp.isna().sum().sum())

### 2.3.10 Cleaning Sisa

In [ ]:
print("--- CLEAN INF DAN MISSING TERSISA ---")

df_previousapp = df_previousapp.replace([np.inf, -np.inf], 0)

num_cols = df_previousapp.select_dtypes(include=[np.number]).columns.tolist()

for col in num_cols:
    missing_before = df_previousapp[col].isna().sum()

    if missing_before > 0:
        median_val = df_previousapp[col].median()
        df_previousapp[col] = df_previousapp[col].fillna(median_val)
        print(f"{col}: missing {missing_before} diisi median {median_val}")

cat_cols = df_previousapp.select_dtypes(include=["object", "category"]).columns.tolist()

for col in cat_cols:
    if str(df_previousapp[col].dtype) == "category":
        if "Unknown" not in df_previousapp[col].cat.categories:
            df_previousapp[col] = df_previousapp[col].cat.add_categories("Unknown")

    df_previousapp[col] = df_previousapp[col].fillna("Unknown")

print("Total missing setelah cleanup:", df_previousapp.isna().sum().sum())

In [ ]:
print("--- FINAL CHECK PREVIOUS APPLICATION ---")

print("Dimensi akhir:", df_previousapp.shape)
print("Total missing:", df_previousapp.isna().sum().sum())
print("Duplicate rows:", df_previousapp.duplicated().sum())

if "SK_ID_PREV" in df_previousapp.columns:
    print("Duplicate SK_ID_PREV:", df_previousapp["SK_ID_PREV"].duplicated().sum())

if "SK_ID_CURR" in df_previousapp.columns:
    print("Unique SK_ID_CURR:", df_previousapp["SK_ID_CURR"].nunique())

print("\nMissing summary:")
print(df_previousapp.isna().sum().sort_values(ascending=False).to_string())

print("\nDtypes:")
print(df_previousapp.dtypes)

### 2.3.11 Aggregation

#### 2.3.11.1 Early Check

In [ ]:
print("--- CEK AWAL SEBELUM AGREGASI PREVIOUS APPLICATION ---")

print("Dimensi df_previousapp:", df_previousapp.shape)
print("Unique SK_ID_CURR:", df_previousapp["SK_ID_CURR"].nunique())
print("Duplicate SK_ID_CURR:", df_previousapp["SK_ID_CURR"].duplicated().sum())
print("Unique SK_ID_PREV:", df_previousapp["SK_ID_PREV"].nunique())
print("Duplicate SK_ID_PREV:", df_previousapp["SK_ID_PREV"].duplicated().sum())
print("Total missing:", df_previousapp.isna().sum().sum())

#### 2.3.11.2 Basic Aggregation

In [ ]:
print("--- BASIC PREVIOUS APPLICATION AGGREGATION ---")

previous_base_agg = df_previousapp.groupby("SK_ID_CURR").agg(
    PREV_APPLICATION_COUNT=("SK_ID_PREV", "count"),
    PREV_UNIQUE_CONTRACT_TYPE_COUNT=("NAME_CONTRACT_TYPE", "nunique"),
    PREV_UNIQUE_CONTRACT_STATUS_COUNT=("NAME_CONTRACT_STATUS", "nunique"),
    PREV_UNIQUE_CLIENT_TYPE_COUNT=("NAME_CLIENT_TYPE", "nunique"),
    PREV_UNIQUE_CHANNEL_TYPE_COUNT=("CHANNEL_TYPE", "nunique"),
    PREV_UNIQUE_PRODUCT_COMBINATION_COUNT=("PRODUCT_COMBINATION", "nunique")
).reset_index()

print("Dimensi previous_base_agg:", previous_base_agg.shape)
previous_base_agg.head()

#### 2.3.11.3 Numeric Aggregation

In [ ]:
print("--- TAHAP 2: NUMERIC AGGREGATION ---")

num_agg_cols = [
    "AMT_ANNUITY",
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE",
    "HOUR_APPR_PROCESS_START",
    "NFLAG_LAST_APPL_IN_DAY",
    "RATE_DOWN_PAYMENT",
    "DAYS_DECISION",
    "SELLERPLACE_AREA",
    "CNT_PAYMENT",
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
    "NFLAG_INSURED_ON_APPROVAL",
    "PREV_CREDIT_APPLICATION_DIFF",
    "PREV_CREDIT_APPLICATION_RATIO",
    "PREV_DOWN_PAYMENT_APPLICATION_RATIO",
    "PREV_ANNUITY_CREDIT_RATIO",
    "PREV_CREDIT_ANNUITY_RATIO",
    "PREV_GOODS_CREDIT_RATIO",
    "PREV_PLANNED_DURATION",
    "PREV_ACTUAL_DURATION",
    "PREV_DURATION_DIFF"
]

num_agg_cols = [col for col in num_agg_cols if col in df_previousapp.columns]

previous_num_agg = df_previousapp.groupby("SK_ID_CURR")[num_agg_cols].agg(
    ["mean", "max", "min", "sum"]
)

previous_num_agg.columns = [
    "PREV_" + col.upper() + "_" + stat.upper()
    for col, stat in previous_num_agg.columns
]

previous_num_agg = previous_num_agg.reset_index()

print("Jumlah kolom numerik diagregasi:", len(num_agg_cols))
print("Dimensi previous_num_agg:", previous_num_agg.shape)
previous_num_agg.head()

#### 2.3.11.4 Missing Flag Aggregation

In [ ]:
print("--- MISSING FLAG AGGREGATION ---")

missing_flag_cols = [
    col for col in df_previousapp.columns
    if col.endswith("_MISSING")
]

previous_missing_agg = df_previousapp.groupby("SK_ID_CURR")[missing_flag_cols].agg(
    ["sum", "mean"]
)

previous_missing_agg.columns = [
    "PREV_" + col.upper() + "_" + stat.upper()
    for col, stat in previous_missing_agg.columns
]

previous_missing_agg = previous_missing_agg.reset_index()

print("Missing flag cols:", missing_flag_cols)
print("Dimensi previous_missing_agg:", previous_missing_agg.shape)
previous_missing_agg.head()

#### 2.3.11.5 Anomaly Flag Aggregation

In [ ]:
print("--- ANOMALY FLAG AGGREGATION ---")

anom_flag_cols = [
    col for col in df_previousapp.columns
    if col.endswith("_ANOM")
]

previous_anom_agg = df_previousapp.groupby("SK_ID_CURR")[anom_flag_cols].agg(
    ["sum", "mean"]
)

previous_anom_agg.columns = [
    "PREV_" + col.upper() + "_" + stat.upper()
    for col, stat in previous_anom_agg.columns
]

previous_anom_agg = previous_anom_agg.reset_index()

print("Anomaly flag cols:", anom_flag_cols)
print("Dimensi previous_anom_agg:", previous_anom_agg.shape)
previous_anom_agg.head()

#### 2.3.11.6 Status Flag Aggregation

In [ ]:
print("--- STATUS FLAG AGGREGATION ---")

status_flag_cols = [
    "PREV_IS_APPROVED",
    "PREV_IS_REFUSED",
    "PREV_IS_CANCELED",
    "PREV_IS_UNUSED_OFFER"
]

status_flag_cols = [
    col for col in status_flag_cols
    if col in df_previousapp.columns
]

previous_status_agg = df_previousapp.groupby("SK_ID_CURR")[status_flag_cols].agg(
    ["sum", "mean"]
)

previous_status_agg.columns = [
    col.upper() + "_" + stat.upper()
    for col, stat in previous_status_agg.columns
]

previous_status_agg = previous_status_agg.reset_index()

print("Dimensi previous_status_agg:", previous_status_agg.shape)
previous_status_agg.head()

#### 2.3.11.7 Categorical Dummy Aggregation

In [ ]:
print("--- CATEGORICAL DUMMY AGGREGATION ---")

cat_agg_cols = [
    "NAME_CONTRACT_TYPE",
    "NAME_CONTRACT_STATUS",
    "NAME_CLIENT_TYPE",
    "NAME_PORTFOLIO",
    "NAME_PRODUCT_TYPE",
    "CHANNEL_TYPE",
    "NAME_YIELD_GROUP",
    "PRODUCT_COMBINATION"
]

cat_agg_cols = [
    col for col in cat_agg_cols
    if col in df_previousapp.columns
]

previous_cat_aggs = []

for col in cat_agg_cols:
    print(f"Processing categorical column: {col}")

    dummy = pd.get_dummies(
        df_previousapp[["SK_ID_CURR", col]],
        columns=[col],
        prefix="PREV_" + col
    )

    dummy_agg = dummy.groupby("SK_ID_CURR").sum().reset_index()

    previous_cat_aggs.append(dummy_agg)

print("Jumlah categorical agg:", len(previous_cat_aggs))

#### 2.3.11.8 Approved Application Aggregation

In [ ]:

print("--- APPROVED PREVIOUS APPLICATION AGGREGATION ---")

prev_approved = df_previousapp[
    df_previousapp["NAME_CONTRACT_STATUS"] == "Approved"
].copy()

print("Jumlah record Approved:", prev_approved.shape[0])

approved_num_cols = [
    "AMT_ANNUITY",
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE",
    "CNT_PAYMENT",
    "DAYS_DECISION",
    "PREV_CREDIT_APPLICATION_DIFF",
    "PREV_CREDIT_APPLICATION_RATIO",
    "PREV_DOWN_PAYMENT_APPLICATION_RATIO",
    "PREV_ANNUITY_CREDIT_RATIO",
    "PREV_CREDIT_ANNUITY_RATIO",
    "PREV_GOODS_CREDIT_RATIO"
]

approved_num_cols = [
    col for col in approved_num_cols
    if col in prev_approved.columns
]

previous_approved_agg = prev_approved.groupby("SK_ID_CURR")[approved_num_cols].agg(
    ["mean", "max", "min", "sum"]
)

previous_approved_agg.columns = [
    "PREV_APPROVED_" + col.upper() + "_" + stat.upper()
    for col, stat in previous_approved_agg.columns
]

previous_approved_agg = previous_approved_agg.reset_index()

print("Dimensi previous_approved_agg:", previous_approved_agg.shape)
previous_approved_agg.head()

2.3.11.9 Refused Application Aggregation

In [ ]:
print("--- TAHAP 8: REFUSED PREVIOUS APPLICATION AGGREGATION ---")

prev_refused = df_previousapp[
    df_previousapp["NAME_CONTRACT_STATUS"] == "Refused"
].copy()

print("Jumlah record Refused:", prev_refused.shape[0])

refused_num_cols = [
    "AMT_ANNUITY",
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE",
    "CNT_PAYMENT",
    "DAYS_DECISION",
    "PREV_CREDIT_APPLICATION_DIFF",
    "PREV_CREDIT_APPLICATION_RATIO"
]

refused_num_cols = [
    col for col in refused_num_cols
    if col in prev_refused.columns
]

previous_refused_agg = prev_refused.groupby("SK_ID_CURR")[refused_num_cols].agg(
    ["mean", "max", "min", "sum"]
)

previous_refused_agg.columns = [
    "PREV_REFUSED_" + col.upper() + "_" + stat.upper()
    for col, stat in previous_refused_agg.columns
]

previous_refused_agg = previous_refused_agg.reset_index()

print("Dimensi previous_refused_agg:", previous_refused_agg.shape)
previous_refused_agg.head()

#### 2.3.11.9 Last Previous Application

In [ ]:
print("--- TAHAP 9: LAST PREVIOUS APPLICATION FEATURES ---")

prev_last = (
    df_previousapp.sort_values("DAYS_DECISION", ascending=False)
    .groupby("SK_ID_CURR")
    .head(1)
    .copy()
)

last_cols = [
    "SK_ID_CURR",
    "AMT_ANNUITY",
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE",
    "CNT_PAYMENT",
    "DAYS_DECISION",
    "PREV_CREDIT_APPLICATION_DIFF",
    "PREV_CREDIT_APPLICATION_RATIO",
    "PREV_DOWN_PAYMENT_APPLICATION_RATIO",
    "PREV_ANNUITY_CREDIT_RATIO",
    "PREV_CREDIT_ANNUITY_RATIO",
    "PREV_GOODS_CREDIT_RATIO",
    "PREV_IS_APPROVED",
    "PREV_IS_REFUSED",
    "PREV_IS_CANCELED",
    "PREV_IS_UNUSED_OFFER"
]

last_cols = [
    col for col in last_cols
    if col in prev_last.columns
]

previous_last_agg = prev_last[last_cols].copy()

rename_cols = {
    col: "PREV_LAST_" + col.upper()
    for col in previous_last_agg.columns
    if col != "SK_ID_CURR"
}

previous_last_agg = previous_last_agg.rename(columns=rename_cols)

print("Dimensi previous_last_agg:", previous_last_agg.shape)
previous_last_agg.head()

#### 2.3.11.10 Merge All Aggregation

In [ ]:
print("--- MERGE SEMUA PREVIOUS APPLICATION AGGREGATION ---")

previous_aggs = [
    previous_base_agg,
    previous_num_agg,
    previous_missing_agg,
    previous_anom_agg,
    previous_status_agg,
    previous_approved_agg,
    previous_refused_agg,
    previous_last_agg
]

# Tambahkan categorical aggregations
previous_aggs.extend(previous_cat_aggs)

df_previousapp_agg = previous_aggs[0]

for agg_df in previous_aggs[1:]:
    df_previousapp_agg = df_previousapp_agg.merge(
        agg_df,
        on="SK_ID_CURR",
        how="left"
    )

df_previousapp_agg = df_previousapp_agg.replace([np.inf, -np.inf], 0).fillna(0)

print("Dimensi df_previousapp_agg:", df_previousapp_agg.shape)
print("Unique SK_ID_CURR:", df_previousapp_agg["SK_ID_CURR"].nunique())
print("Duplicate SK_ID_CURR:", df_previousapp_agg["SK_ID_CURR"].duplicated().sum())
print("Total missing:", df_previousapp_agg.isna().sum().sum())

df_previousapp_agg.head()

In [ ]:
df_previousapp_agg.describe()

### 2.3.12 Post Aggregation Feature

In [ ]:
print("--- TAHAP 11: POST-AGGREGATION FEATURES ---")

if {
    "PREV_IS_APPROVED_SUM",
    "PREV_APPLICATION_COUNT"
}.issubset(df_previousapp_agg.columns):
    df_previousapp_agg["PREV_APPROVAL_RATE"] = (
        df_previousapp_agg["PREV_IS_APPROVED_SUM"] /
        (df_previousapp_agg["PREV_APPLICATION_COUNT"] + 1)
    )

if {
    "PREV_IS_REFUSED_SUM",
    "PREV_APPLICATION_COUNT"
}.issubset(df_previousapp_agg.columns):
    df_previousapp_agg["PREV_REFUSAL_RATE"] = (
        df_previousapp_agg["PREV_IS_REFUSED_SUM"] /
        (df_previousapp_agg["PREV_APPLICATION_COUNT"] + 1)
    )

if {
    "PREV_IS_CANCELED_SUM",
    "PREV_APPLICATION_COUNT"
}.issubset(df_previousapp_agg.columns):
    df_previousapp_agg["PREV_CANCEL_RATE"] = (
        df_previousapp_agg["PREV_IS_CANCELED_SUM"] /
        (df_previousapp_agg["PREV_APPLICATION_COUNT"] + 1)
    )

if {
    "PREV_AMT_CREDIT_SUM",
    "PREV_AMT_APPLICATION_SUM"
}.issubset(df_previousapp_agg.columns):
    df_previousapp_agg["PREV_TOTAL_CREDIT_APPLICATION_RATIO"] = (
        df_previousapp_agg["PREV_AMT_CREDIT_SUM"] /
        (df_previousapp_agg["PREV_AMT_APPLICATION_SUM"] + 1)
    )

if {
    "PREV_AMT_DOWN_PAYMENT_SUM",
    "PREV_AMT_APPLICATION_SUM"
}.issubset(df_previousapp_agg.columns):
    df_previousapp_agg["PREV_TOTAL_DOWN_PAYMENT_APPLICATION_RATIO"] = (
        df_previousapp_agg["PREV_AMT_DOWN_PAYMENT_SUM"] /
        (df_previousapp_agg["PREV_AMT_APPLICATION_SUM"] + 1)
    )

df_previousapp_agg = df_previousapp_agg.replace([np.inf, -np.inf], 0).fillna(0)

print("Dimensi setelah post-agg features:", df_previousapp_agg.shape)
print("Total missing:", df_previousapp_agg.isna().sum().sum())

df_previousapp_agg.head()

### 2.3.13 Reduce Memory

In [ ]:
print("--- OPTIMASI MEMORI PREVIOUS APPLICATION AGG ---")

df_previousapp_agg = reduce_mem_usage_safe(df_previousapp_agg)

df_previousapp_agg.info()

### 2.3.14 Final Check

In [ ]:
print("--- FINAL CHECK PREVIOUS APPLICATION AGG ---")

print("Dimensi df_previousapp_agg:", df_previousapp_agg.shape)
print("Unique SK_ID_CURR:", df_previousapp_agg["SK_ID_CURR"].nunique())
print("Duplicate SK_ID_CURR:", df_previousapp_agg["SK_ID_CURR"].duplicated().sum())
print("Total missing:", df_previousapp_agg.isna().sum().sum())

display(df_previousapp_agg.head())

### 2.3.15 Merge to Application Data

In [ ]:
print("--- MERGE PREVIOUS APPLICATION AGG KE APPLICATION FULL ---")

print("Dimensi df_application_full sebelum merge:", df_application_full.shape)
print("Dimensi df_previousapp_agg:", df_previousapp_agg.shape)

df_application_full = df_application_full.merge(
    df_previousapp_agg,
    on="SK_ID_CURR",
    how="left"
)

previous_agg_cols = [
    col for col in df_previousapp_agg.columns
    if col != "SK_ID_CURR"
]

df_application_full[previous_agg_cols] = df_application_full[previous_agg_cols].fillna(0)

print("Dimensi df_application_full setelah merge:", df_application_full.shape)
print("Total missing setelah merge:", df_application_full.isna().sum().sum())
print("Duplicate SK_ID_CURR setelah merge:", df_application_full["SK_ID_CURR"].duplicated().sum())
print("Unique SK_ID_CURR:", df_application_full["SK_ID_CURR"].nunique())

In [ ]:
num_cols = df_application_full.select_dtypes(include=[np.number]).columns

print("Jumlah infinite:", np.isinf(df_application_full[num_cols]).sum().sum())

## 2.4 Bureau Balance

### 2.4.1 Early Check

In [ ]:
print("--- CEK AWAL: BUREAU BALANCE ---")

print("Dimensi awal:", df_bureaubalance.shape)

print("\nInfo:")
df_bureaubalance.info()

print("\nMissing value:")
missing_bb = (
    pd.DataFrame({
        "missing_count": df_bureaubalance.isna().sum(),
        "missing_pct": df_bureaubalance.isna().mean() * 100
    })
    .sort_values("missing_count", ascending=False)
)

print(missing_bb.to_string())

print("\nDuplicate rows:", df_bureaubalance.duplicated().sum())

if "SK_ID_BUREAU" in df_bureaubalance.columns:
    print("Unique SK_ID_BUREAU:", df_bureaubalance["SK_ID_BUREAU"].nunique())

if "STATUS" in df_bureaubalance.columns:
    print("\nDistribusi STATUS:")
    print(df_bureaubalance["STATUS"].value_counts(dropna=False))

### 2.4.2 Memory Optimization


In [ ]:
print("--- OPTIMASI MEMORI AWAL BUREAU BALANCE ---")

df_bureaubalance["SK_ID_BUREAU"] = pd.to_numeric(
    df_bureaubalance["SK_ID_BUREAU"],
    downcast="integer"
)

df_bureaubalance["MONTHS_BALANCE"] = pd.to_numeric(
    df_bureaubalance["MONTHS_BALANCE"],
    downcast="integer"
)

df_bureaubalance["STATUS"] = df_bureaubalance["STATUS"].astype("category")

df_bureaubalance.info()

### 2.4.3 Feature Engineering Sederhana: Status

In [ ]:
print("--- STATUS NUMERIC DAN FLAGS ---")

status_map = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "C": 0,
    "X": 0
}

df_bureaubalance["BB_STATUS_NUM"] = (
    df_bureaubalance["STATUS"]
    .astype(str)
    .map(status_map)
    .fillna(0)
    .astype("int8")
)

df_bureaubalance["BB_IS_DPD"] = (
    df_bureaubalance["STATUS"].isin(["1", "2", "3", "4", "5"])
).astype("int8")

df_bureaubalance["BB_IS_BAD_DPD"] = (
    df_bureaubalance["STATUS"].isin(["3", "4", "5"])
).astype("int8")

df_bureaubalance["BB_IS_CLOSED"] = (
    df_bureaubalance["STATUS"] == "C"
).astype("int8")

df_bureaubalance["BB_IS_UNKNOWN"] = (
    df_bureaubalance["STATUS"] == "X"
).astype("int8")

# Status individual flags
for s in ["0", "1", "2", "3", "4", "5", "C", "X"]:
    df_bureaubalance[f"BB_STATUS_{s}"] = (
        df_bureaubalance["STATUS"] == s
    ).astype("int8")


print("\nMemory setelah flags:")
df_bureaubalance.info()

In [ ]:
df_bureaubalance.head()

### 2.4.4 Aggregation

In [ ]:
print("--- AGREGASI PER SK_ID_BUREAU ---")

bb_agg_dict = {
    "MONTHS_BALANCE": ["count", "min", "max", "mean"],
    "BB_STATUS_NUM": ["mean", "max", "sum"],
    "BB_IS_DPD": ["sum", "mean"],
    "BB_IS_BAD_DPD": ["sum", "mean"],
    "BB_IS_CLOSED": ["sum", "mean"],
    "BB_IS_UNKNOWN": ["sum", "mean"],
    "BB_STATUS_0": ["sum"],
    "BB_STATUS_1": ["sum"],
    "BB_STATUS_2": ["sum"],
    "BB_STATUS_3": ["sum"],
    "BB_STATUS_4": ["sum"],
    "BB_STATUS_5": ["sum"],
    "BB_STATUS_C": ["sum"],
    "BB_STATUS_X": ["sum"]
}

df_bb_bureau_agg = df_bureaubalance.groupby("SK_ID_BUREAU").agg(bb_agg_dict)

df_bb_bureau_agg.columns = [
    "BB_" + col.upper() + "_" + stat.upper()
    for col, stat in df_bb_bureau_agg.columns
]

df_bb_bureau_agg = df_bb_bureau_agg.reset_index()

print("Dimensi df_bb_bureau_agg:", df_bb_bureau_agg.shape)
print("Duplicate SK_ID_BUREAU:", df_bb_bureau_agg["SK_ID_BUREAU"].duplicated().sum())
print("Total missing:", df_bb_bureau_agg.isna().sum().sum())

df_bb_bureau_agg.head()

### 2.4.5 Ratio

In [ ]:
print("--- TAHAP 4: STATUS RATIO PER SK_ID_BUREAU ---")

status_sum_cols = [
    "BB_BB_STATUS_0_SUM",
    "BB_BB_STATUS_1_SUM",
    "BB_BB_STATUS_2_SUM",
    "BB_BB_STATUS_3_SUM",
    "BB_BB_STATUS_4_SUM",
    "BB_BB_STATUS_5_SUM",
    "BB_BB_STATUS_C_SUM",
    "BB_BB_STATUS_X_SUM"
]

status_sum_cols = [
    col for col in status_sum_cols
    if col in df_bb_bureau_agg.columns
]

for col in status_sum_cols:
    ratio_col = col.replace("_SUM", "_RATIO")
    df_bb_bureau_agg[ratio_col] = (
        df_bb_bureau_agg[col] /
        (df_bb_bureau_agg["BB_MONTHS_BALANCE_COUNT"] + 1)
    )

df_bb_bureau_agg = df_bb_bureau_agg.replace([np.inf, -np.inf], 0).fillna(0)

print("Dimensi setelah ratio:", df_bb_bureau_agg.shape)
df_bb_bureau_agg.head()

### 2.4.6 Latest Status

In [ ]:
print("--- LATEST STATUS PER SK_ID_BUREAU ---")

bb_latest = (
    df_bureaubalance
    .sort_values(["SK_ID_BUREAU", "MONTHS_BALANCE"], ascending=[True, False])
    .groupby("SK_ID_BUREAU")
    .head(1)
    [["SK_ID_BUREAU", "MONTHS_BALANCE", "BB_STATUS_NUM", "BB_IS_DPD", "BB_IS_BAD_DPD", "BB_IS_CLOSED", "BB_IS_UNKNOWN"]]
    .copy()
)

bb_latest = bb_latest.rename(columns={
    "MONTHS_BALANCE": "BB_LATEST_MONTHS_BALANCE",
    "BB_STATUS_NUM": "BB_LATEST_STATUS_NUM",
    "BB_IS_DPD": "BB_LATEST_IS_DPD",
    "BB_IS_BAD_DPD": "BB_LATEST_IS_BAD_DPD",
    "BB_IS_CLOSED": "BB_LATEST_IS_CLOSED",
    "BB_IS_UNKNOWN": "BB_LATEST_IS_UNKNOWN"
})

print("Dimensi bb_latest:", bb_latest.shape)
print("Duplicate SK_ID_BUREAU:", bb_latest["SK_ID_BUREAU"].duplicated().sum())

df_bb_bureau_agg = df_bb_bureau_agg.merge(
    bb_latest,
    on="SK_ID_BUREAU",
    how="left"
)

df_bb_bureau_agg = df_bb_bureau_agg.fillna(0)

print("Dimensi setelah latest merge:", df_bb_bureau_agg.shape)
print("Total missing:", df_bb_bureau_agg.isna().sum().sum())

### 2.4.7 Merge Mapping

In [ ]:
print("--- MERGE DENGAN MAPPING BUREAU ---")

bureau_mapping = df_bureau[["SK_ID_BUREAU", "SK_ID_CURR"]].drop_duplicates()

print("Dimensi bureau_mapping:", bureau_mapping.shape)
print("Duplicate SK_ID_BUREAU mapping:", bureau_mapping["SK_ID_BUREAU"].duplicated().sum())

df_bb_with_curr = df_bb_bureau_agg.merge(
    bureau_mapping,
    on="SK_ID_BUREAU",
    how="left"
)

print("Dimensi df_bb_with_curr:", df_bb_with_curr.shape)
print("Missing SK_ID_CURR:", df_bb_with_curr["SK_ID_CURR"].isna().sum())

### 2.4.8 Agregasi Bureau Balance ke SK_ID_CURR

In [ ]:
print("--- CEK DF_BB_WITH_CURR ---")

print("Dimensi df_bb_with_curr:", df_bb_with_curr.shape)
print("Missing SK_ID_CURR:", df_bb_with_curr["SK_ID_CURR"].isna().sum())
print("Duplicate SK_ID_BUREAU:", df_bb_with_curr["SK_ID_BUREAU"].duplicated().sum())
print("Unique SK_ID_CURR:", df_bb_with_curr["SK_ID_CURR"].nunique())
print("Total missing:", df_bb_with_curr.isna().sum().sum())

missing_curr = df_bb_with_curr["SK_ID_CURR"].isna().sum()

if missing_curr > 0:
    print(f"Menghapus {missing_curr} baris karena tidak punya SK_ID_CURR")
    df_bb_with_curr = df_bb_with_curr.dropna(subset=["SK_ID_CURR"])

df_bb_with_curr["SK_ID_CURR"] = df_bb_with_curr["SK_ID_CURR"].astype("int32")

print("Missing SK_ID_CURR setelah handling:", df_bb_with_curr["SK_ID_CURR"].isna().sum())

In [ ]:
print("--- AGREGASI BUREAU BALANCE KE SK_ID_CURR ---")

bb_feature_cols = [
    col for col in df_bb_with_curr.columns
    if col not in ["SK_ID_BUREAU", "SK_ID_CURR"]
]

df_bureau_balance_agg = df_bb_with_curr.groupby("SK_ID_CURR")[bb_feature_cols].agg(
    ["mean", "max", "min", "sum"]
)

df_bureau_balance_agg.columns = [
    "BBCURR_" + col.upper() + "_" + stat.upper()
    for col, stat in df_bureau_balance_agg.columns
]

df_bureau_balance_agg = df_bureau_balance_agg.reset_index()

df_bureau_balance_agg = (
    df_bureau_balance_agg
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

print("Dimensi df_bureau_balance_agg:", df_bureau_balance_agg.shape)
print("Unique SK_ID_CURR:", df_bureau_balance_agg["SK_ID_CURR"].nunique())
print("Duplicate SK_ID_CURR:", df_bureau_balance_agg["SK_ID_CURR"].duplicated().sum())
print("Total missing:", df_bureau_balance_agg.isna().sum().sum())

In [ ]:
print("--- POST-AGGREGATION FEATURES BUREAU BALANCE ---")

if {
    "BBCURR_BB_BB_IS_DPD_SUM_SUM",
    "BBCURR_BB_MONTHS_BALANCE_COUNT_SUM"
}.issubset(df_bureau_balance_agg.columns):
    df_bureau_balance_agg["BB_TOTAL_DPD_RATIO"] = (
        df_bureau_balance_agg["BBCURR_BB_BB_IS_DPD_SUM_SUM"] /
        (df_bureau_balance_agg["BBCURR_BB_MONTHS_BALANCE_COUNT_SUM"] + 1)
    )

if {
    "BBCURR_BB_BB_IS_BAD_DPD_SUM_SUM",
    "BBCURR_BB_MONTHS_BALANCE_COUNT_SUM"
}.issubset(df_bureau_balance_agg.columns):
    df_bureau_balance_agg["BB_TOTAL_BAD_DPD_RATIO"] = (
        df_bureau_balance_agg["BBCURR_BB_BB_IS_BAD_DPD_SUM_SUM"] /
        (df_bureau_balance_agg["BBCURR_BB_MONTHS_BALANCE_COUNT_SUM"] + 1)
    )

if {
    "BBCURR_BB_BB_IS_CLOSED_SUM_SUM",
    "BBCURR_BB_MONTHS_BALANCE_COUNT_SUM"
}.issubset(df_bureau_balance_agg.columns):
    df_bureau_balance_agg["BB_TOTAL_CLOSED_RATIO"] = (
        df_bureau_balance_agg["BBCURR_BB_BB_IS_CLOSED_SUM_SUM"] /
        (df_bureau_balance_agg["BBCURR_BB_MONTHS_BALANCE_COUNT_SUM"] + 1)
    )

if {
    "BBCURR_BB_BB_IS_UNKNOWN_SUM_SUM",
    "BBCURR_BB_MONTHS_BALANCE_COUNT_SUM"
}.issubset(df_bureau_balance_agg.columns):
    df_bureau_balance_agg["BB_TOTAL_UNKNOWN_RATIO"] = (
        df_bureau_balance_agg["BBCURR_BB_BB_IS_UNKNOWN_SUM_SUM"] /
        (df_bureau_balance_agg["BBCURR_BB_MONTHS_BALANCE_COUNT_SUM"] + 1)
    )

df_bureau_balance_agg = (
    df_bureau_balance_agg
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

print("Dimensi setelah post-agg:", df_bureau_balance_agg.shape)
print("Total missing:", df_bureau_balance_agg.isna().sum().sum())

In [ ]:
print("--- OPTIMASI MEMORI BUREAU BALANCE AGG ---")

df_bureau_balance_agg = reduce_mem_usage_safe(df_bureau_balance_agg)

df_bureau_balance_agg.info()

In [ ]:
print("--- FINAL CHECK BUREAU BALANCE AGG ---")

print("Dimensi:", df_bureau_balance_agg.shape)
print("Unique SK_ID_CURR:", df_bureau_balance_agg["SK_ID_CURR"].nunique())
print("Duplicate SK_ID_CURR:", df_bureau_balance_agg["SK_ID_CURR"].duplicated().sum())
print("Total missing:", df_bureau_balance_agg.isna().sum().sum())

num_cols = df_bureau_balance_agg.select_dtypes(include=[np.number]).columns
print("Jumlah infinite:", np.isinf(df_bureau_balance_agg[num_cols]).sum().sum())

In [ ]:
print("--- MERGE BUREAU BALANCE AGG KE APPLICATION FULL ---")

print("Dimensi df_application_full sebelum merge:", df_application_full.shape)
print("Dimensi df_bureau_balance_agg:", df_bureau_balance_agg.shape)

df_application_full = df_application_full.merge(
    df_bureau_balance_agg,
    on="SK_ID_CURR",
    how="left"
)

bb_agg_cols = [
    col for col in df_bureau_balance_agg.columns
    if col != "SK_ID_CURR"
]

df_application_full[bb_agg_cols] = df_application_full[bb_agg_cols].fillna(0)

print("Dimensi df_application_full setelah merge:", df_application_full.shape)
print("Total missing setelah merge:", df_application_full.isna().sum().sum())
print("Duplicate SK_ID_CURR setelah merge:", df_application_full["SK_ID_CURR"].duplicated().sum())

num_cols = df_application_full.select_dtypes(include=[np.number]).columns
print("Jumlah infinite setelah merge:", np.isinf(df_application_full[num_cols]).sum().sum())

## 2.5 POS Cash Balance


### 2.5.1 Early Check


In [ ]:
print("--- CEK AWAL: POS_CASH_balance ---")

print("Dimensi awal:", df_poscashbalance.shape)
df_poscashbalance.info()

missing_pos = (
    pd.DataFrame({
        "missing_count": df_poscashbalance.isna().sum(),
        "missing_pct": df_poscashbalance.isna().mean() * 100
    })
    .sort_values("missing_count", ascending=False)
)

print(missing_pos.to_string())

print("Duplicate rows:", df_poscashbalance.duplicated().sum())
print("Unique SK_ID_CURR:", df_poscashbalance["SK_ID_CURR"].nunique())
print("Unique SK_ID_PREV:", df_poscashbalance["SK_ID_PREV"].nunique())

### 2.5.2 Cleaning


In [ ]:
print("--- CLEANING POS_CASH_balance ---")

df_poscashbalance = df_poscashbalance.drop_duplicates()

cat_cols = df_poscashbalance.select_dtypes(include=["object", "category"]).columns.tolist()

for col in cat_cols:
    df_poscashbalance[col] = df_poscashbalance[col].fillna("Unknown")

num_cols = df_poscashbalance.select_dtypes(include=[np.number]).columns.tolist()

for col in num_cols:
    if df_poscashbalance[col].isna().sum() > 0:
        df_poscashbalance[col] = df_poscashbalance[col].fillna(df_poscashbalance[col].median())

df_poscashbalance = df_poscashbalance.replace([np.inf, -np.inf], 0).fillna(0)

print("Dimensi setelah cleaning:", df_poscashbalance.shape)
print("Total missing:", df_poscashbalance.isna().sum().sum())

### 2.5.3 Feature Engineering


In [ ]:
print("--- FEATURE POS_CASH_balance ---")

if "SK_DPD" in df_poscashbalance.columns:
    df_poscashbalance["POS_IS_DPD"] = (df_poscashbalance["SK_DPD"] > 0).astype("int8")
    df_poscashbalance["POS_IS_BAD_DPD"] = (df_poscashbalance["SK_DPD"] >= 30).astype("int8")

if "SK_DPD_DEF" in df_poscashbalance.columns:
    df_poscashbalance["POS_IS_DPD_DEF"] = (df_poscashbalance["SK_DPD_DEF"] > 0).astype("int8")
    df_poscashbalance["POS_IS_BAD_DPD_DEF"] = (df_poscashbalance["SK_DPD_DEF"] >= 30).astype("int8")

if "NAME_CONTRACT_STATUS" in df_poscashbalance.columns:
    df_poscashbalance["POS_IS_ACTIVE"] = (
        df_poscashbalance["NAME_CONTRACT_STATUS"] == "Active"
    ).astype("int8")

    df_poscashbalance["POS_IS_COMPLETED"] = (
        df_poscashbalance["NAME_CONTRACT_STATUS"] == "Completed"
    ).astype("int8")

    df_poscashbalance["POS_IS_SIGNED"] = (
        df_poscashbalance["NAME_CONTRACT_STATUS"] == "Signed"
    ).astype("int8")

### 2.5.4 Aggregation


In [ ]:
print("--- AGREGASI POS_CASH_balance KE SK_ID_CURR ---")

pos_num_cols = [
    "MONTHS_BALANCE",
    "CNT_INSTALMENT",
    "CNT_INSTALMENT_FUTURE",
    "SK_DPD",
    "SK_DPD_DEF",
    "POS_IS_DPD",
    "POS_IS_BAD_DPD",
    "POS_IS_DPD_DEF",
    "POS_IS_BAD_DPD_DEF",
    "POS_IS_ACTIVE",
    "POS_IS_COMPLETED",
    "POS_IS_SIGNED"
]

pos_num_cols = [col for col in pos_num_cols if col in df_poscashbalance.columns]

df_poscash_agg = df_poscashbalance.groupby("SK_ID_CURR")[pos_num_cols].agg(
    ["mean", "max", "min", "sum"]
)

df_poscash_agg.columns = [
    "POS_" + col.upper() + "_" + stat.upper()
    for col, stat in df_poscash_agg.columns
]

df_poscash_agg = df_poscash_agg.reset_index()

pos_count_agg = df_poscashbalance.groupby("SK_ID_CURR").agg(
    POS_RECORD_COUNT=("SK_ID_PREV", "count"),
    POS_UNIQUE_PREV_COUNT=("SK_ID_PREV", "nunique")
).reset_index()

df_poscash_agg = df_poscash_agg.merge(
    pos_count_agg,
    on="SK_ID_CURR",
    how="left"
)

df_poscash_agg = df_poscash_agg.replace([np.inf, -np.inf], 0).fillna(0)

print("Dimensi df_poscash_agg:", df_poscash_agg.shape)
print("Duplicate SK_ID_CURR:", df_poscash_agg["SK_ID_CURR"].duplicated().sum())
print("Total missing:", df_poscash_agg.isna().sum().sum())

### 2.5.5 Merge to Application Data


In [ ]:
print("--- MERGE POS_CASH AGG KE APPLICATION FULL ---")

print("Dimensi sebelum merge:", df_application_full.shape)
print("Dimensi df_poscash_agg:", df_poscash_agg.shape)

df_application_full = df_application_full.merge(
    df_poscash_agg,
    on="SK_ID_CURR",
    how="left"
)

pos_cols = [col for col in df_poscash_agg.columns if col != "SK_ID_CURR"]
df_application_full[pos_cols] = df_application_full[pos_cols].fillna(0)

num_cols = df_application_full.select_dtypes(include=[np.number]).columns

print("Dimensi setelah merge:", df_application_full.shape)
print("Total missing:", df_application_full.isna().sum().sum())
print("Duplicate SK_ID_CURR:", df_application_full["SK_ID_CURR"].duplicated().sum())
print("Infinite:", np.isinf(df_application_full[num_cols]).sum().sum())

## 2.6 Installments Payments


### 2.6.1 Early Check


In [ ]:
print("--- CEK AWAL: INSTALLMENTS PAYMENTS ---")

print("Dimensi awal:", df_installments.shape)
df_installments.info()

missing_inst = (
    pd.DataFrame({
        "missing_count": df_installments.isna().sum(),
        "missing_pct": df_installments.isna().mean() * 100
    })
    .sort_values("missing_count", ascending=False)
)

print(missing_inst.to_string())

print("Duplicate rows:", df_installments.duplicated().sum())
print("Unique SK_ID_CURR:", df_installments["SK_ID_CURR"].nunique())
print("Unique SK_ID_PREV:", df_installments["SK_ID_PREV"].nunique())

### 2.6.2 Cleaning + Feature Engineering


In [ ]:
print("--- CLEANING + FEATURE INSTALLMENTS ---")

df_installments = df_installments.drop_duplicates()

num_cols = df_installments.select_dtypes(include=[np.number]).columns.tolist()

for col in num_cols:
    if df_installments[col].isna().sum() > 0:
        df_installments[col] = df_installments[col].fillna(df_installments[col].median())

df_installments = df_installments.replace([np.inf, -np.inf], 0).fillna(0)

# Payment behavior
df_installments["INST_PAYMENT_DIFF"] = (
    df_installments["AMT_PAYMENT"] - df_installments["AMT_INSTALMENT"]
)

df_installments["INST_PAYMENT_RATIO"] = (
    df_installments["AMT_PAYMENT"] / (df_installments["AMT_INSTALMENT"] + 1)
)

df_installments["INST_DAYS_DIFF"] = (
    df_installments["DAYS_ENTRY_PAYMENT"] - df_installments["DAYS_INSTALMENT"]
)

# Telat bayar kalau payment entry lebih besar dari due date
df_installments["INST_IS_LATE"] = (
    df_installments["INST_DAYS_DIFF"] > 0
).astype("int8")

df_installments["INST_IS_UNDERPAID"] = (
    df_installments["AMT_PAYMENT"] < df_installments["AMT_INSTALMENT"]
).astype("int8")

df_installments["INST_IS_OVERPAID"] = (
    df_installments["AMT_PAYMENT"] > df_installments["AMT_INSTALMENT"]
).astype("int8")

df_installments = df_installments.replace([np.inf, -np.inf], 0).fillna(0)

print("Dimensi setelah feature:", df_installments.shape)
print("Total missing:", df_installments.isna().sum().sum())

### 2.6.3 Aggregation


In [ ]:
print("--- AGREGASI INSTALLMENTS KE SK_ID_CURR ---")

inst_num_cols = [
    "NUM_INSTALMENT_VERSION",
    "NUM_INSTALMENT_NUMBER",
    "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT",
    "AMT_PAYMENT",
    "INST_PAYMENT_DIFF",
    "INST_PAYMENT_RATIO",
    "INST_DAYS_DIFF",
    "INST_IS_LATE",
    "INST_IS_UNDERPAID",
    "INST_IS_OVERPAID"
]

inst_num_cols = [col for col in inst_num_cols if col in df_installments.columns]

df_installments_agg = df_installments.groupby("SK_ID_CURR")[inst_num_cols].agg(
    ["mean", "max", "min", "sum"]
)

df_installments_agg.columns = [
    "INST_" + col.upper() + "_" + stat.upper()
    for col, stat in df_installments_agg.columns
]

df_installments_agg = df_installments_agg.reset_index()

inst_count_agg = df_installments.groupby("SK_ID_CURR").agg(
    INST_RECORD_COUNT=("SK_ID_PREV", "count"),
    INST_UNIQUE_PREV_COUNT=("SK_ID_PREV", "nunique")
).reset_index()

df_installments_agg = df_installments_agg.merge(
    inst_count_agg,
    on="SK_ID_CURR",
    how="left"
)

if {
    "INST_INST_IS_LATE_SUM",
    "INST_RECORD_COUNT"
}.issubset(df_installments_agg.columns):
    df_installments_agg["INST_LATE_PAYMENT_RATIO"] = (
        df_installments_agg["INST_INST_IS_LATE_SUM"] /
        (df_installments_agg["INST_RECORD_COUNT"] + 1)
    )

if {
    "INST_INST_IS_UNDERPAID_SUM",
    "INST_RECORD_COUNT"
}.issubset(df_installments_agg.columns):
    df_installments_agg["INST_UNDERPAID_RATIO"] = (
        df_installments_agg["INST_INST_IS_UNDERPAID_SUM"] /
        (df_installments_agg["INST_RECORD_COUNT"] + 1)
    )

df_installments_agg = df_installments_agg.replace([np.inf, -np.inf], 0).fillna(0)

print("Dimensi df_installments_agg:", df_installments_agg.shape)
print("Duplicate SK_ID_CURR:", df_installments_agg["SK_ID_CURR"].duplicated().sum())
print("Total missing:", df_installments_agg.isna().sum().sum())

### 2.6.4 Merge to Application Data


In [ ]:
print("--- MERGE INSTALLMENTS AGG KE APPLICATION FULL ---")

print("Dimensi sebelum merge:", df_application_full.shape)
print("Dimensi df_installments_agg:", df_installments_agg.shape)

df_application_full = df_application_full.merge(
    df_installments_agg,
    on="SK_ID_CURR",
    how="left"
)

inst_cols = [col for col in df_installments_agg.columns if col != "SK_ID_CURR"]
df_application_full[inst_cols] = df_application_full[inst_cols].fillna(0)

num_cols = df_application_full.select_dtypes(include=[np.number]).columns

print("Dimensi setelah merge:", df_application_full.shape)
print("Total missing:", df_application_full.isna().sum().sum())
print("Duplicate SK_ID_CURR:", df_application_full["SK_ID_CURR"].duplicated().sum())
print("Infinite:", np.isinf(df_application_full[num_cols]).sum().sum())

## 2.7 Credit Card Balance


### 2.7.1 Early Check


In [ ]:
print("--- CEK AWAL: CREDIT CARD BALANCE ---")

print("Dimensi awal:", df_credit_card_balance.shape)
df_credit_card_balance.info()

missing_cc = (
    pd.DataFrame({
        "missing_count": df_credit_card_balance.isna().sum(),
        "missing_pct": df_credit_card_balance.isna().mean() * 100
    })
    .sort_values("missing_count", ascending=False)
)

print(missing_cc.to_string())

print("Duplicate rows:", df_credit_card_balance.duplicated().sum())
print("Unique SK_ID_CURR:", df_credit_card_balance["SK_ID_CURR"].nunique())
print("Unique SK_ID_PREV:", df_credit_card_balance["SK_ID_PREV"].nunique())

### 2.7.2 Cleaning + Feature Engineering


In [ ]:
print("--- CLEANING + FEATURE CREDIT CARD ---")

df_credit_card_balance = df_credit_card_balance.drop_duplicates()

cat_cols = df_credit_card_balance.select_dtypes(include=["object", "category"]).columns.tolist()

for col in cat_cols:
    df_credit_card_balance[col] = df_credit_card_balance[col].fillna("Unknown")

num_cols = df_credit_card_balance.select_dtypes(include=[np.number]).columns.tolist()

for col in num_cols:
    if df_credit_card_balance[col].isna().sum() > 0:
        df_credit_card_balance[col] = df_credit_card_balance[col].fillna(df_credit_card_balance[col].median())

df_credit_card_balance = df_credit_card_balance.replace([np.inf, -np.inf], 0).fillna(0)

if {"AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL"}.issubset(df_credit_card_balance.columns):
    df_credit_card_balance["CC_LIMIT_USE_RATIO"] = (
        df_credit_card_balance["AMT_BALANCE"] /
        (df_credit_card_balance["AMT_CREDIT_LIMIT_ACTUAL"] + 1)
    )

if {"AMT_PAYMENT_TOTAL_CURRENT", "AMT_INST_MIN_REGULARITY"}.issubset(df_credit_card_balance.columns):
    df_credit_card_balance["CC_PAYMENT_MIN_RATIO"] = (
        df_credit_card_balance["AMT_PAYMENT_TOTAL_CURRENT"] /
        (df_credit_card_balance["AMT_INST_MIN_REGULARITY"] + 1)
    )

if "SK_DPD" in df_credit_card_balance.columns:
    df_credit_card_balance["CC_IS_DPD"] = (df_credit_card_balance["SK_DPD"] > 0).astype("int8")
    df_credit_card_balance["CC_IS_BAD_DPD"] = (df_credit_card_balance["SK_DPD"] >= 30).astype("int8")

if "SK_DPD_DEF" in df_credit_card_balance.columns:
    df_credit_card_balance["CC_IS_DPD_DEF"] = (df_credit_card_balance["SK_DPD_DEF"] > 0).astype("int8")
    df_credit_card_balance["CC_IS_BAD_DPD_DEF"] = (df_credit_card_balance["SK_DPD_DEF"] >= 30).astype("int8")

df_credit_card_balance = df_credit_card_balance.replace([np.inf, -np.inf], 0).fillna(0)

print("Dimensi setelah feature:", df_credit_card_balance.shape)
print("Total missing:", df_credit_card_balance.isna().sum().sum())

### 2.7.3 Aggregation


In [ ]:
print("--- AGREGASI CREDIT CARD KE SK_ID_CURR ---")

cc_num_cols = [
    col for col in df_credit_card_balance.select_dtypes(include=[np.number]).columns
    if col not in ["SK_ID_CURR", "SK_ID_PREV"]
]

df_creditcard_agg = df_credit_card_balance.groupby("SK_ID_CURR")[cc_num_cols].agg(
    ["mean", "max", "min", "sum"]
)

df_creditcard_agg.columns = [
    "CC_" + col.upper() + "_" + stat.upper()
    for col, stat in df_creditcard_agg.columns
]

df_creditcard_agg = df_creditcard_agg.reset_index()

cc_count_agg = df_credit_card_balance.groupby("SK_ID_CURR").agg(
    CC_RECORD_COUNT=("SK_ID_PREV", "count"),
    CC_UNIQUE_PREV_COUNT=("SK_ID_PREV", "nunique")
).reset_index()

df_creditcard_agg = df_creditcard_agg.merge(
    cc_count_agg,
    on="SK_ID_CURR",
    how="left"
)

df_creditcard_agg = df_creditcard_agg.replace([np.inf, -np.inf], 0).fillna(0)

print("Dimensi df_creditcard_agg:", df_creditcard_agg.shape)
print("Duplicate SK_ID_CURR:", df_creditcard_agg["SK_ID_CURR"].duplicated().sum())
print("Total missing:", df_creditcard_agg.isna().sum().sum())

### 2.7.4 Merge to Application Data


In [ ]:
print("--- MERGE CREDIT CARD AGG KE APPLICATION FULL ---")

print("Dimensi sebelum merge:", df_application_full.shape)
print("Dimensi df_creditcard_agg:", df_creditcard_agg.shape)

df_application_full = df_application_full.merge(
    df_creditcard_agg,
    on="SK_ID_CURR",
    how="left"
)

cc_cols = [col for col in df_creditcard_agg.columns if col != "SK_ID_CURR"]
df_application_full[cc_cols] = df_application_full[cc_cols].fillna(0)

num_cols = df_application_full.select_dtypes(include=[np.number]).columns

print("Dimensi setelah merge:", df_application_full.shape)
print("Total missing:", df_application_full.isna().sum().sum())
print("Duplicate SK_ID_CURR:", df_application_full["SK_ID_CURR"].duplicated().sum())
print("Infinite:", np.isinf(df_application_full[num_cols]).sum().sum())

## 2.8 Final Check All Merged Data


### 2.8.1 Sanity Check


In [ ]:
print("--- FINAL CHECK SEMUA DATA TERGABUNG ---")

print("Shape final:", df_application_full.shape)
print("Total missing:", df_application_full.isna().sum().sum())
print("Duplicate SK_ID_CURR:", df_application_full["SK_ID_CURR"].duplicated().sum())

num_cols = df_application_full.select_dtypes(include=[np.number]).columns
print("Infinite:", np.isinf(df_application_full[num_cols]).sum().sum())

print("\nDtypes:")
print(df_application_full.dtypes.value_counts())

df_application_full.info()

## 3. Modeling


### 3.0 Import Modeling Libraries


In [ ]:
import os
import re
import sys
import joblib
import subprocess
import matplotlib.pyplot as plt

try:
    import lightgbm as lgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm", "-q"])
    import lightgbm as lgb

from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

### 3.1 Final Check Data Gabungan


In [ ]:
print("--- FINAL CHECK DATA GABUNGAN ---")

print("Shape:", df_application_full.shape)
print("Total missing:", df_application_full.isna().sum().sum())
print("Duplicate SK_ID_CURR:", df_application_full["SK_ID_CURR"].duplicated().sum())

num_cols = df_application_full.select_dtypes(include=[np.number]).columns
print("Jumlah infinite:", np.isinf(df_application_full[num_cols]).sum().sum())

print("\nDistribusi TARGET:")
print(df_application_full["TARGET"].value_counts())

print("\nDistribusi TARGET (%):")
print(df_application_full["TARGET"].value_counts(normalize=True) * 100)

### 3.2 Final Cleaning Sebelum Modeling


In [ ]:
print("--- FINAL CLEANING BEFORE MODELING ---")

df_model = df_application_full.copy()

# Bersihkan infinite di kolom numerik
num_cols = df_model.select_dtypes(include=[np.number]).columns
df_model[num_cols] = df_model[num_cols].replace([np.inf, -np.inf], 0)

# Fill missing numerik
df_model[num_cols] = df_model[num_cols].fillna(0)

# Fill missing kategorikal
cat_cols = df_model.select_dtypes(include=["object", "category"]).columns

for col in cat_cols:
    if str(df_model[col].dtype) == "category":
        if "Unknown" not in df_model[col].cat.categories:
            df_model[col] = df_model[col].cat.add_categories("Unknown")
    df_model[col] = df_model[col].fillna("Unknown")

# Boolean ke int8
bool_cols = df_model.select_dtypes(include=["bool"]).columns

for col in bool_cols:
    df_model[col] = df_model[col].astype("int8")

print("Total missing setelah final cleaning:", df_model.isna().sum().sum())

num_cols = df_model.select_dtypes(include=[np.number]).columns
print("Jumlah infinite setelah final cleaning:", np.isinf(df_model[num_cols]).sum().sum())

print("Shape df_model:", df_model.shape)

### 3.3 Bersihkan Nama Kolom


In [ ]:
def clean_column_names(df):
    df = df.copy()

    new_cols = []
    seen = {}

    for col in df.columns:
        clean_col = str(col)
        clean_col = re.sub(r"[^A-Za-z0-9_]+", "_", clean_col)
        clean_col = re.sub(r"_+", "_", clean_col)
        clean_col = clean_col.strip("_")

        if clean_col == "":
            clean_col = "COL"

        if clean_col in seen:
            seen[clean_col] += 1
            clean_col = f"{clean_col}_{seen[clean_col]}"
        else:
            seen[clean_col] = 0

        new_cols.append(clean_col)

    df.columns = new_cols
    return df


df_model = clean_column_names(df_model)

print("Shape setelah clean column names:", df_model.shape)
print("Duplicate columns:", df_model.columns.duplicated().sum())

### 3.4 Split X dan y


In [ ]:
print("--- SPLIT X DAN y ---")

y = df_model["TARGET"].astype("int8")
X = df_model.drop(columns=["TARGET", "SK_ID_CURR"])

print("Shape X:", X.shape)
print("Shape y:", y.shape)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Jumlah fitur numerik:", len(num_cols))
print("Jumlah fitur kategorikal:", len(cat_cols))
print("Kolom kategorikal:", cat_cols)

### 3.5 Train Validation Test Split


In [ ]:
print("--- TRAIN VALIDATION TEST SPLIT ---")

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,
    y_temp,
    test_size=0.2,
    random_state=42,
    stratify=y_temp
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test :", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\ny_valid distribution:")
print(y_valid.value_counts(normalize=True) * 100)

print("\ny_test distribution:")
print(y_test.value_counts(normalize=True) * 100)

### 3.6 Fungsi Evaluasi


In [ ]:
def evaluate_model(y_true, proba, threshold=0.5, model_name="Model"):
    pred = (proba >= threshold).astype(int)

    auc = roc_auc_score(y_true, proba)
    ap = average_precision_score(y_true, proba)

    acc = accuracy_score(y_true, pred)
    precision = precision_score(y_true, pred, zero_division=0)
    recall = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)

    print(f"--- {model_name} Evaluation ---")
    print(f"Threshold          : {threshold:.3f}")
    print(f"ROC-AUC            : {auc:.5f}")
    print(f"Average Precision  : {ap:.5f}")
    print(f"Accuracy           : {acc:.5f}")
    print(f"Precision          : {precision:.5f}")
    print(f"Recall             : {recall:.5f}")
    print(f"F1-score           : {f1:.5f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, pred))

    print("\nClassification Report:")
    print(classification_report(y_true, pred, zero_division=0))

    return {
        "model": model_name,
        "threshold": threshold,
        "roc_auc": auc,
        "average_precision": ap,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


def find_best_threshold(y_true, proba):
    thresholds = np.arange(0.05, 0.95, 0.01)
    results = []

    for threshold in thresholds:
        pred = (proba >= threshold).astype(int)

        results.append({
            "threshold": threshold,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0)
        })

    threshold_df = pd.DataFrame(results).sort_values("f1", ascending=False)

    return threshold_df

### 3.7 Prepare Data untuk LightGBM


In [ ]:
print("--- PREPARE DATA FOR LIGHTGBM ---")

X_train_lgbm = X_train.copy()
X_valid_lgbm = X_valid.copy()
X_test_lgbm = X_test.copy()

for col in cat_cols:
    X_train_lgbm[col] = X_train_lgbm[col].astype("category")
    train_categories = X_train_lgbm[col].cat.categories
    X_valid_lgbm[col] = pd.Categorical(X_valid_lgbm[col], categories=train_categories)
    X_test_lgbm[col] = pd.Categorical(X_test_lgbm[col], categories=train_categories)

print("X_train_lgbm:", X_train_lgbm.shape)
print("X_valid_lgbm:", X_valid_lgbm.shape)
print("X_test_lgbm :", X_test_lgbm.shape)

### 3.8 Train LightGBM Baseline


In [ ]:
print("--- TRAINING LIGHTGBM BASELINE ---")

model_lgbm = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)

try:
    model_lgbm.fit(
        X_train_lgbm,
        y_train,
        eval_set=[(X_valid_lgbm, y_valid)],
        eval_metric="auc",
        categorical_feature=cat_cols,
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),
            lgb.log_evaluation(period=100)
        ]
    )
except TypeError:
    print("Fallback fit tanpa callbacks.")
    model_lgbm.fit(
        X_train_lgbm,
        y_train,
        categorical_feature=cat_cols
    )

valid_proba_lgbm = model_lgbm.predict_proba(X_valid_lgbm)[:, 1]
test_proba_lgbm = model_lgbm.predict_proba(X_test_lgbm)[:, 1]

lgbm_valid_05 = evaluate_model(
    y_valid,
    valid_proba_lgbm,
    threshold=0.5,
    model_name="LightGBM Validation @0.5"
)

lgbm_test_05 = evaluate_model(
    y_test,
    test_proba_lgbm,
    threshold=0.5,
    model_name="LightGBM Test @0.5"
)

### 3.9 Best Threshold LightGBM


In [ ]:
print("--- BEST THRESHOLD LIGHTGBM (TUNED ON VALIDATION) ---")

threshold_lgbm_df = find_best_threshold(y_valid, valid_proba_lgbm)

display(threshold_lgbm_df.head(10))

best_threshold_lgbm = threshold_lgbm_df.iloc[0]["threshold"]

print("Best threshold LightGBM:", best_threshold_lgbm)

lgbm_valid_best = evaluate_model(
    y_valid,
    valid_proba_lgbm,
    threshold=best_threshold_lgbm,
    model_name="LightGBM Validation Best Threshold"
)

lgbm_test_best = evaluate_model(
    y_test,
    test_proba_lgbm,
    threshold=best_threshold_lgbm,
    model_name="LightGBM Test Best Threshold"
)

### 3.10 Logistic Regression Benchmark


In [ ]:
print("--- TRAINING LOGISTIC REGRESSION BENCHMARK ---")

numeric_features = num_cols
categorical_features = cat_cols

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler(with_mean=False))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_logreg = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

model_logreg = Pipeline(
    steps=[
        ("preprocessor", preprocessor_logreg),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="saga",
            n_jobs=-1,
            random_state=42
        ))
    ]
)

model_logreg.fit(X_train, y_train)

valid_proba_logreg = model_logreg.predict_proba(X_valid)[:, 1]
test_proba_logreg = model_logreg.predict_proba(X_test)[:, 1]

logreg_valid_05 = evaluate_model(
    y_valid,
    valid_proba_logreg,
    threshold=0.5,
    model_name="Logistic Regression Validation @0.5"
)

logreg_test_05 = evaluate_model(
    y_test,
    test_proba_logreg,
    threshold=0.5,
    model_name="Logistic Regression Test @0.5"
)

### 3.11 Best Threshold Logistic Regression


In [ ]:
print("--- BEST THRESHOLD LOGISTIC REGRESSION (TUNED ON VALIDATION) ---")

threshold_logreg_df = find_best_threshold(y_valid, valid_proba_logreg)

display(threshold_logreg_df.head(10))

best_threshold_logreg = threshold_logreg_df.iloc[0]["threshold"]

print("Best threshold Logistic Regression:", best_threshold_logreg)

logreg_valid_best = evaluate_model(
    y_valid,
    valid_proba_logreg,
    threshold=best_threshold_logreg,
    model_name="Logistic Regression Validation Best Threshold"
)

logreg_test_best = evaluate_model(
    y_test,
    test_proba_logreg,
    threshold=best_threshold_logreg,
    model_name="Logistic Regression Test Best Threshold"
)

### 3.12 Validation Model Comparison


In [ ]:
print("--- VALIDATION MODEL COMPARISON ---")

comparison_df = pd.DataFrame([
    lgbm_valid_05,
    lgbm_valid_best,
    logreg_valid_05,
    logreg_valid_best
])

comparison_df = comparison_df.sort_values("roc_auc", ascending=False)

display(comparison_df)

### 3.13 ROC Curve


In [ ]:
print("--- ROC CURVE (TEST SET) ---")

fpr_lgbm, tpr_lgbm, _ = roc_curve(y_test, test_proba_lgbm)
fpr_logreg, tpr_logreg, _ = roc_curve(y_test, test_proba_logreg)

auc_lgbm = roc_auc_score(y_test, test_proba_lgbm)
auc_logreg = roc_auc_score(y_test, test_proba_logreg)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lgbm, tpr_lgbm, label=f"LightGBM Test AUC = {auc_lgbm:.4f}")
plt.plot(fpr_logreg, tpr_logreg, label=f"LogReg Test AUC = {auc_logreg:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Test Set")
plt.legend()
plt.show()

### 3.14 Precision Recall Curve


In [ ]:
print("--- PRECISION RECALL CURVE (TEST SET) ---")

precision_lgbm, recall_lgbm, _ = precision_recall_curve(y_test, test_proba_lgbm)
precision_logreg, recall_logreg, _ = precision_recall_curve(y_test, test_proba_logreg)

ap_lgbm = average_precision_score(y_test, test_proba_lgbm)
ap_logreg = average_precision_score(y_test, test_proba_logreg)

plt.figure(figsize=(8, 6))
plt.plot(recall_lgbm, precision_lgbm, label=f"LightGBM Test AP = {ap_lgbm:.4f}")
plt.plot(recall_logreg, precision_logreg, label=f"LogReg Test AP = {ap_logreg:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Test Set")
plt.legend()
plt.show()

### 3.15 Feature Importance LightGBM


In [ ]:
print("--- FEATURE IMPORTANCE LIGHTGBM ---")

feature_importance = pd.DataFrame({
    "feature": X_train_lgbm.columns,
    "importance": model_lgbm.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(50))
top_n = 30

plt.figure(figsize=(10, 8))
plt.barh(
    feature_importance.head(top_n)["feature"][::-1],
    feature_importance.head(top_n)["importance"][::-1]
)
plt.xlabel("Importance")
plt.title(f"Top {top_n} Feature Importance - LightGBM")
plt.show()

### 3.16 Cek Overfitting LightGBM


In [ ]:
print("--- CEK OVERFITTING LIGHTGBM ---")

train_proba_lgbm = model_lgbm.predict_proba(X_train_lgbm)[:, 1]

train_auc_lgbm = roc_auc_score(y_train, train_proba_lgbm)
valid_auc_lgbm = roc_auc_score(y_valid, valid_proba_lgbm)
test_auc_lgbm = roc_auc_score(y_test, test_proba_lgbm)

print("Train AUC LightGBM:", train_auc_lgbm)
print("Valid AUC LightGBM:", valid_auc_lgbm)
print("Test AUC LightGBM :", test_auc_lgbm)
print("Train-Valid Gap:", train_auc_lgbm - valid_auc_lgbm)
print("Train-Test Gap :", train_auc_lgbm - test_auc_lgbm)

### 3.17 Simple Ensemble LGBM + LogReg


In [ ]:
print("--- SIMPLE ENSEMBLE: LIGHTGBM + LOGISTIC REGRESSION ---")

ensemble_results = []

weights = np.arange(0.0, 1.05, 0.05)

for w in weights:
    ensemble_valid_proba = (w * valid_proba_lgbm) + ((1 - w) * valid_proba_logreg)
    auc = roc_auc_score(y_valid, ensemble_valid_proba)
    ap = average_precision_score(y_valid, ensemble_valid_proba)

    ensemble_results.append({
        "weight_lgbm": w,
        "weight_logreg": 1 - w,
        "roc_auc": auc,
        "average_precision": ap
    })

ensemble_df = pd.DataFrame(ensemble_results).sort_values("roc_auc", ascending=False)

display(ensemble_df.head(10))

best_w = ensemble_df.iloc[0]["weight_lgbm"]

ensemble_valid_proba_best = (
    best_w * valid_proba_lgbm
    + (1 - best_w) * valid_proba_logreg
)

ensemble_test_proba_best = (
    best_w * test_proba_lgbm
    + (1 - best_w) * test_proba_logreg
)

print("Best LightGBM weight:", best_w)
print("Best Logistic weight:", 1 - best_w)

ensemble_valid_05 = evaluate_model(
    y_valid,
    ensemble_valid_proba_best,
    threshold=0.5,
    model_name="Ensemble Validation @0.5"
)

ensemble_test_05 = evaluate_model(
    y_test,
    ensemble_test_proba_best,
    threshold=0.5,
    model_name="Ensemble Test @0.5"
)

### 3.18 Best Threshold Ensemble


In [ ]:
print("--- BEST THRESHOLD ENSEMBLE (TUNED ON VALIDATION) ---")

threshold_ensemble_df = find_best_threshold(y_valid, ensemble_valid_proba_best)

display(threshold_ensemble_df.head(10))

best_threshold_ensemble = threshold_ensemble_df.iloc[0]["threshold"]

ensemble_valid_best = evaluate_model(
    y_valid,
    ensemble_valid_proba_best,
    threshold=best_threshold_ensemble,
    model_name="Ensemble Validation Best Threshold"
)

ensemble_test_best = evaluate_model(
    y_test,
    ensemble_test_proba_best,
    threshold=best_threshold_ensemble,
    model_name="Ensemble Test Best Threshold"
)

### 3.19 Final Test Model Comparison


In [ ]:
print("--- FINAL TEST MODEL COMPARISON ---")

final_comparison = pd.DataFrame([
    lgbm_test_05,
    lgbm_test_best,
    logreg_test_05,
    logreg_test_best,
    ensemble_test_05,
    ensemble_test_best
])

final_comparison = final_comparison.sort_values("roc_auc", ascending=False)

display(final_comparison)

### 3.20 Save Model dan Fitur


In [ ]:
print("--- SAVE MODEL DAN FITUR ---")

artifact_dir = "/content/drive/MyDrive/Home Credit Risk/model_artifacts"

if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(artifact_dir, exist_ok=True)
else:
    artifact_dir = "."

artifact_paths = {
    "model_lgbm_baseline.pkl": model_lgbm,
    "model_logreg_benchmark.pkl": model_logreg,
    "lgbm_feature_names.pkl": X_train_lgbm.columns.tolist(),
    "categorical_columns.pkl": cat_cols,
    "numeric_columns.pkl": num_cols,
    "best_threshold_lgbm.pkl": best_threshold_lgbm,
    "best_threshold_logreg.pkl": best_threshold_logreg,
    "best_threshold_ensemble.pkl": best_threshold_ensemble,
    "best_ensemble_weight_lgbm.pkl": best_w,
}

for filename, artifact in artifact_paths.items():
    joblib.dump(artifact, os.path.join(artifact_dir, filename))

df_model.to_pickle(os.path.join(artifact_dir, "df_model_final.pkl"))
feature_importance.to_csv(os.path.join(artifact_dir, "feature_importance_lgbm.csv"), index=False)
comparison_df.to_csv(os.path.join(artifact_dir, "validation_model_comparison.csv"), index=False)
final_comparison.to_csv(os.path.join(artifact_dir, "test_model_comparison.csv"), index=False)
threshold_lgbm_df.to_csv(os.path.join(artifact_dir, "threshold_lgbm_validation.csv"), index=False)
threshold_logreg_df.to_csv(os.path.join(artifact_dir, "threshold_logreg_validation.csv"), index=False)
threshold_ensemble_df.to_csv(os.path.join(artifact_dir, "threshold_ensemble_validation.csv"), index=False)
ensemble_df.to_csv(os.path.join(artifact_dir, "ensemble_weight_validation.csv"), index=False)

print("Artifact directory:", artifact_dir)
print("Saved files:")
for filename in sorted(os.listdir(artifact_dir)):
    print("-", filename)

### 3.21 Kesimpulan Workflow Modeling


In [ ]:
print("--- KESIMPULAN WORKFLOW MODELING ---")
print("1. LightGBM baseline result")
print("2. Logistic Regression benchmark result")
print("3. Threshold dan ensemble weight dituning di validation set")
print("4. Final comparison dilaporkan di test set")
print("5. ROC curve dan precision-recall curve memakai test set")
print("6. Feature importance LightGBM tersedia")
print("7. Model, threshold, fitur, dan comparison tersimpan")
print("8. Untuk validasi produksi, refactor 2.x agar preprocessing fit hanya pada train split")